# PolicyDelta

Semantic Privacy Policy Change Detector

Run the cells in order using **Runtime → Run all** in Google Colab.

In [ ]:
%pip install -q beautifulsoup4==4.14.3 PyMuPDF==1.26.7 scikit-learn==1.7.2 sentence-transformers==6.2.0 huggingface-hub==1.33.0 "gradio>=6.0,<7"

## Configuration

In [ ]:
import hashlib
import importlib.util
import json
import platform
import tempfile
import random
import re
import sys
import unittest
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Literal

from bs4 import BeautifulSoup, NavigableString, Tag
import pymupdf


class PolicyDeltaError(ValueError):
    """Invalid policy data or application configuration."""


class EvidenceError(PolicyDeltaError):
    """Source evidence could not be verified."""


@dataclass(frozen=True, slots=True)
class Settings:
    seed: int = 42
    max_document_chars: int = 250_000
    max_pdf_pages: int = 200
    max_pdf_bytes: int = 25_000_000

    def __post_init__(self) -> None:
        if type(self.seed) is not int or not 0 <= self.seed < 2**32:
            raise PolicyDeltaError("Seed must be a nonnegative 32-bit integer.")
        if type(self.max_document_chars) is not int or self.max_document_chars < 1:
            raise PolicyDeltaError("Document character limit must be positive.")
        if type(self.max_pdf_pages) is not int or self.max_pdf_pages < 1:
            raise PolicyDeltaError("PDF page limit must be positive.")
        if type(self.max_pdf_bytes) is not int or self.max_pdf_bytes < 1:
            raise PolicyDeltaError("PDF file size limit must be positive.")


SETTINGS = Settings()


def initialize(settings: Settings = SETTINGS) -> dict[str, str | int]:
    if sys.version_info < (3, 10):
        raise RuntimeError("Python 3.10 or later is required.")
    random.seed(settings.seed)
    try:
        colab = importlib.util.find_spec("google.colab") is not None
    except (ImportError, ModuleNotFoundError, ValueError):
        colab = False
    return {
        "python": platform.python_version(),
        "environment": "colab" if colab else "python",
        "seed": settings.seed,
    }


RUNTIME = initialize()
print(f"PolicyDelta | Python {RUNTIME['python']} | {RUNTIME['environment']}")

## Document model

In [ ]:
@dataclass(frozen=True, slots=True)
class PolicyDocument:
    document_id: str
    role: Literal["old", "new"]
    text: str
    sha256: str

    def __post_init__(self) -> None:
        if not isinstance(self.document_id, str) or not self.document_id.strip():
            raise PolicyDeltaError("Document ID must be nonempty.")
        if self.role not in ("old", "new"):
            raise PolicyDeltaError("Document role must be 'old' or 'new'.")
        if not isinstance(self.text, str) or not self.text.strip():
            raise PolicyDeltaError("Policy text must be nonempty.")
        if self.sha256 != hashlib.sha256(self.text.encode("utf-8")).hexdigest():
            raise EvidenceError("Document checksum does not match its content.")

    @classmethod
    def from_text(
        cls,
        text: str,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        if not isinstance(text, str):
            raise PolicyDeltaError("Policy content must be text.")
        if len(text) > settings.max_document_chars:
            raise PolicyDeltaError("Document exceeds the configured size limit.")
        digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
        return cls(document_id if document_id is not None else f"{role}-{digest[:16]}",
                   role, text, digest)

    @classmethod
    def from_txt(
        cls,
        path: str | Path,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        """Read UTF-8 policy text without changing its line endings."""
        try:
            source = Path(path)
        except (TypeError, ValueError) as exc:
            raise PolicyDeltaError("A valid text-file path is required.") from exc
        if source.suffix.lower() != ".txt":
            raise PolicyDeltaError("Only .txt files are supported.")
        try:
            with source.open(encoding="utf-8-sig", newline="") as file:
                text = file.read(settings.max_document_chars + 1)
        except (OSError, UnicodeError) as exc:
            raise PolicyDeltaError(f"Unable to read UTF-8 text from '{source.name}'.") from exc
        return cls.from_text(text, role=role, document_id=document_id, settings=settings)

    @classmethod
    def from_html(
        cls,
        html: str,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        """Extract readable text from HTML with stable paragraph boundaries."""
        if not isinstance(html, str):
            raise PolicyDeltaError("HTML content must be text.")
        if len(html) > settings.max_document_chars * 8:
            raise PolicyDeltaError("HTML input exceeds the configured size limit.")

        soup = BeautifulSoup(html, "html.parser")
        root = soup.find("main") or soup.find("article") or soup.body or soup
        excluded = {"script", "style", "noscript", "template", "svg", "nav",
                    "header", "footer", "aside", "form"}
        chrome = {"nav", "navigation", "navbar", "menu", "sidebar", "breadcrumb",
                  "cookie-banner", "cookie-notice", "site-header", "site-footer"}
        for tag in list(root.find_all(True)):
            if tag.name is None:
                continue
            identifiers = [str(tag.get("id", "")), *tag.get("class", [])]
            if (tag.name in excluded or tag.has_attr("hidden")
                    or tag.get("aria-hidden") == "true"
                    or re.search(r"(?:display\s*:\s*none|visibility\s*:\s*hidden)",
                                 str(tag.get("style", "")), flags=re.I)
                    or tag.get("role") in {"navigation", "banner", "contentinfo", "search"}
                    or any(value.lower() in chrome for value in identifiers)):
                tag.decompose()

        paragraphs: list[str] = []
        blocks = {"p", "li", "blockquote", "pre", "dt", "dd", "h1", "h2", "h3",
                  "h4", "h5", "h6", "td", "th"}
        containers = blocks | {"div", "section", "article", "main", "ul", "ol",
                               "table", "tbody", "thead", "tr", "dl"}

        def append(text: str) -> None:
            normalized = re.sub(r"\s+", " ", text).strip()
            if normalized:
                paragraphs.append(normalized)

        def collect(tag: Tag) -> None:
            if tag.name in blocks and tag.find(["br", *containers]) is None:
                append(tag.get_text())
                return
            inline: list[str] = []
            for child in tag.children:
                if isinstance(child, NavigableString):
                    inline.append(str(child))
                elif isinstance(child, Tag) and child.name == "br":
                    append("".join(inline))
                    inline.clear()
                elif isinstance(child, Tag) and child.name in containers:
                    append("".join(inline))
                    inline.clear()
                    collect(child)
                elif isinstance(child, Tag):
                    inline.append(child.get_text())
            append("".join(inline))

        collect(root)
        return cls.from_text("\n\n".join(paragraphs), role=role,
                             document_id=document_id, settings=settings)

    @classmethod
    def from_html_file(
        cls,
        path: str | Path,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        """Read a UTF-8 HTML file and extract its policy text."""
        try:
            source = Path(path)
        except (TypeError, ValueError) as exc:
            raise PolicyDeltaError("A valid HTML-file path is required.") from exc
        if source.suffix.lower() not in {".html", ".htm"}:
            raise PolicyDeltaError("Only .html and .htm files are supported.")
        try:
            with source.open(encoding="utf-8-sig", newline="") as file:
                html = file.read(settings.max_document_chars * 8 + 1)
        except (OSError, UnicodeError) as exc:
            raise PolicyDeltaError(f"Unable to read UTF-8 HTML from '{source.name}'.") from exc
        return cls.from_html(html, role=role, document_id=document_id,
                             settings=settings)

    @classmethod
    def from_pdf(
        cls,
        path: str | Path,
        *,
        role: Literal["old", "new"],
        document_id: str | None = None,
        settings: Settings = SETTINGS,
    ) -> "PolicyDocument":
        """Extract text from an unencrypted, text-based PDF."""
        try:
            source = Path(path)
        except (TypeError, ValueError) as exc:
            raise PolicyDeltaError("A valid PDF path is required.") from exc
        if source.suffix.lower() != ".pdf":
            raise PolicyDeltaError("Only .pdf files are supported.")
        try:
            if source.stat().st_size > settings.max_pdf_bytes:
                raise PolicyDeltaError("PDF exceeds the configured file size limit.")
            with pymupdf.open(filename=str(source), filetype="pdf") as pdf:
                if pdf.needs_pass:
                    raise PolicyDeltaError("Password-protected PDFs are not supported.")
                if pdf.page_count > settings.max_pdf_pages:
                    raise PolicyDeltaError("PDF exceeds the configured page limit.")
                pages: list[str] = []
                length = 0
                for page in pdf:
                    text = page.get_text("text", sort=True).strip()
                    if not text:
                        continue
                    length += len(text) + (2 if pages else 0)
                    if length > settings.max_document_chars:
                        raise PolicyDeltaError("Document exceeds the configured size limit.")
                    pages.append(text)
        except (OSError, pymupdf.FileDataError) as exc:
            raise PolicyDeltaError(f"Unable to read PDF '{source.name}'.") from exc
        if not pages:
            raise PolicyDeltaError("PDF contains no extractable text; image-only PDFs require OCR.")
        return cls.from_text("\n\n".join(pages), role=role,
                             document_id=document_id, settings=settings)

    def excerpt(self, start: int, end: int) -> "EvidenceSpan":
        if (type(start) is not int or type(end) is not int
                or start < 0 or end > len(self.text) or start >= end):
            raise EvidenceError("Evidence offsets are outside the source text.")
        return EvidenceSpan(self.document_id, start, end, self.text[start:end])


@dataclass(frozen=True, slots=True)
class EvidenceSpan:
    document_id: str
    start: int
    end: int
    text: str


def verify_evidence(span: EvidenceSpan, document: PolicyDocument) -> None:
    """Confirm that a cited excerpt is an exact slice of the source."""
    if (span.document_id != document.document_id
            or type(span.start) is not int
            or type(span.end) is not int
            or span.start < 0
            or span.start >= span.end
            or span.end > len(document.text)
            or document.text[span.start:span.end] != span.text):
        raise EvidenceError("Evidence does not match the source document.")
    if hashlib.sha256(document.text.encode("utf-8")).hexdigest() != document.sha256:
        raise EvidenceError("Source document integrity check failed.")


def document_json(document: PolicyDocument) -> str:
    return json.dumps(asdict(document), ensure_ascii=False, indent=2)

## Text preprocessing

In [ ]:
import unicodedata


def normalize_policy_text(text: str) -> str:
    """Produce searchable text without changing the stored source."""
    return unicodedata.normalize("NFC", re.sub(r"\s+", " ", text)).strip()


@dataclass(frozen=True, slots=True)
class PolicyClause:
    clause_id: str
    document_id: str
    text: str
    start: int
    end: int
    section: str | None
    paragraph_index: int

    @property
    def normalized_text(self) -> str:
        return normalize_policy_text(self.text)


def _is_heading(line: str, *, isolated: bool) -> bool:
    label = re.sub(r"^(?:#{1,6}\s+|\d+(?:\.\d+)*[.)]?\s+)", "", line).strip()
    if not label or len(label) > 90 or len(label.split()) > 12:
        return False
    if label.endswith((".", "!", "?")):
        return False
    if line.startswith("#") or label.endswith(":") or label.isupper() or label.istitle():
        return True
    return (isolated and label[0].isupper()
            and not re.match(r"^(?:We|You|They|This|These|It)\b", label)
            and not re.search(r"\b(?:is|are|was|were|may|must|will|can|shall|should)\b",
                              label, flags=re.I))


def _sentence_ranges(text: str):
    abbreviations = {"e.g.", "i.e.", "mr.", "mrs.", "ms.", "dr.",
                     "prof.", "inc.", "ltd.", "etc.", "u.s.", "u.k.", "vs."}
    start = 0
    for match in re.finditer(r"[.!?]+(?:[\"'”’)]*)(?=\s|$)", text):
        end = match.end()
        preceding = text[start:end].rstrip('\"\'”’)')
        last = preceding.split()[-1].lower() if preceding.split() else ""
        following = text[end:].lstrip()
        if (last in abbreviations
                and not (last in {"u.s.", "u.k."}
                         and re.match(r"(?:We|Our|You|They|This)\b", following))):
            continue
        if (re.fullmatch(r"(?:\d{1,2}|[A-Za-z])\.", last)
                or (following and not re.match(r"[A-Z0-9\"'“‘(]", following))):
            continue
        if text[start:end].strip():
            yield start, end
        start = end
    if text[start:].strip():
        yield start, len(text)


def segment_policy(document: PolicyDocument) -> list[PolicyClause]:
    """Split policy text into traceable clauses with section context."""
    clauses: list[PolicyClause] = []
    section: str | None = None
    paragraph: tuple[int, int] | None = None
    paragraph_index = 0

    def flush() -> None:
        nonlocal paragraph, paragraph_index
        if paragraph is None:
            return
        left, right = paragraph
        block = document.text[left:right]
        for local_start, local_end in _sentence_ranges(block):
            raw = block[local_start:local_end]
            leading = len(raw) - len(raw.lstrip())
            trailing = len(raw.rstrip())
            start, end = left + local_start + leading, left + local_start + trailing
            if start < end:
                clauses.append(PolicyClause(
                    f"{document.document_id}:clause:{len(clauses) + 1:05d}",
                    document.document_id, document.text[start:end], start, end,
                    section, paragraph_index,
                ))
        paragraph_index += 1
        paragraph = None

    lines = document.text.splitlines(keepends=True)
    offset = 0
    for index, line in enumerate(lines):
        content_end = offset + len(line.rstrip("\r\n"))
        content = document.text[offset:content_end].strip()
        isolated = (index == 0 or not lines[index - 1].strip()
                    or index == len(lines) - 1 or not lines[index + 1].strip())
        if not content:
            flush()
        elif _is_heading(content, isolated=isolated):
            flush()
            section = re.sub(r"^(?:#{1,6}\s+|\d+(?:\.\d+)*[.)]?\s+)",
                             "", content).rstrip(":").strip()
        elif re.match(r"^(?:[-*•]\s+|\d+[.)]\s+)", content):
            flush()
            paragraph = (offset, content_end)
            flush()
        elif paragraph is None:
            paragraph = (offset, content_end)
        else:
            paragraph = (paragraph[0], content_end)
        offset += len(line)
    flush()
    if not clauses:
        raise PolicyDeltaError("Document contains no analyzable clauses.")
    return clauses


## Clause alignment

In [ ]:
from collections import defaultdict, deque
from functools import lru_cache
import math

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


@dataclass(frozen=True, slots=True)
class ClauseMatch:
    old: PolicyClause
    new: PolicyClause
    similarity: float
    method: Literal["exact", "tfidf", "minilm"]


@dataclass(frozen=True, slots=True)
class ClauseAlignment:
    matches: tuple[ClauseMatch, ...]
    unmatched_old: tuple[PolicyClause, ...]
    unmatched_new: tuple[PolicyClause, ...]


def tfidf_similarity(old: list[PolicyClause], new: list[PolicyClause]) -> np.ndarray:
    """Calculate lexical similarity using a shared word and bigram vocabulary."""
    if not old or not new:
        return np.zeros((len(old), len(new)), dtype=float)
    texts = [clause.normalized_text for clause in [*old, *new]]
    try:
        vectors = TfidfVectorizer(ngram_range=(1, 2), token_pattern=r"(?u)\b\w+\b").fit_transform(texts)
    except ValueError as exc:
        if "empty vocabulary" not in str(exc).lower():
            raise
        return np.zeros((len(old), len(new)), dtype=float)
    return cosine_similarity(vectors[:len(old)], vectors[len(old):])


@lru_cache(maxsize=1)
def load_semantic_model():
    """Load the sentence encoder once per notebook session."""
    try:
        from sentence_transformers import SentenceTransformer
        return SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    except Exception as exc:
        raise PolicyDeltaError("Unable to load MiniLM. Check the installation and model download.") from exc


def minilm_similarity(
    old: list[PolicyClause], new: list[PolicyClause], *, encoder=None,
) -> np.ndarray:
    """Compute cosine similarities between contextual clause embeddings."""
    if not old or not new:
        return np.zeros((len(old), len(new)), dtype=float)
    model = encoder if encoder is not None else load_semantic_model()
    texts = [clause.normalized_text for clause in [*old, *new]]
    try:
        vectors = np.asarray(model.encode(
            texts, batch_size=32, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=False,
        ), dtype=float)
    except Exception as exc:
        raise PolicyDeltaError("Unable to encode policy clauses.") from exc
    if (vectors.ndim != 2 or vectors.shape[0] != len(texts)
            or vectors.shape[1] == 0 or not np.isfinite(vectors).all()):
        raise PolicyDeltaError("The embedding model returned invalid vectors.")
    norms = np.linalg.norm(vectors, axis=1)
    if np.any(norms == 0):
        raise PolicyDeltaError("The embedding model returned a zero vector.")
    unit_vectors = vectors / norms[:, None]
    return np.clip(unit_vectors[:len(old)] @ unit_vectors[len(old):].T, -1.0, 1.0)


_PROVISIONAL_MATCH_THRESHOLDS = {"tfidf": 0.40, "minilm": 0.65}
CALIBRATED_MATCH_THRESHOLDS: dict[str, float] = {}


def alignment_threshold(method: str) -> float:
    """Use an active validated threshold, or the documented provisional fallback."""
    if method not in _PROVISIONAL_MATCH_THRESHOLDS:
        raise PolicyDeltaError("Alignment method must be tfidf or minilm.")
    return CALIBRATED_MATCH_THRESHOLDS.get(method, _PROVISIONAL_MATCH_THRESHOLDS[method])


@dataclass(frozen=True, slots=True)
class _PreparedAlignment:
    old: tuple[PolicyClause, ...]
    new: tuple[PolicyClause, ...]
    exact: tuple[tuple[int, int], ...]
    old_indices: tuple[int, ...]
    new_indices: tuple[int, ...]
    scores: np.ndarray


def _prepare_alignment(old: PolicyDocument, new: PolicyDocument, method: str, encoder=None) -> _PreparedAlignment:
    old_clauses, new_clauses = tuple(segment_policy(old)), tuple(segment_policy(new))
    available: dict[str, deque[int]] = defaultdict(deque)
    for index, clause in enumerate(new_clauses):
        available[clause.normalized_text.casefold()].append(index)

    exact = []
    used_old, used_new = set(), set()
    for index, clause in enumerate(old_clauses):
        positions = available[clause.normalized_text.casefold()]
        if positions:
            new_index = positions.popleft()
            exact.append((index, new_index))
            used_old.add(index)
            used_new.add(new_index)

    old_indices = tuple(i for i in range(len(old_clauses)) if i not in used_old)
    new_indices = tuple(i for i in range(len(new_clauses)) if i not in used_new)
    remaining_old = [old_clauses[i] for i in old_indices]
    remaining_new = [new_clauses[i] for i in new_indices]
    if method == "tfidf":
        scores = tfidf_similarity(remaining_old, remaining_new)
    else:
        scores = minilm_similarity(remaining_old, remaining_new, encoder=encoder)
    if scores.shape != (len(old_indices), len(new_indices)) or not np.isfinite(scores).all():
        raise PolicyDeltaError("Clause similarity returned invalid scores.")
    return _PreparedAlignment(old_clauses, new_clauses, tuple(exact),
                              old_indices, new_indices, scores)


def _assign_alignment(prepared: _PreparedAlignment, method: str,
                      threshold: float, candidates_per_clause: int) -> ClauseAlignment:
    matches = [ClauseMatch(prepared.old[i], prepared.new[j], 1.0, "exact")
               for i, j in prepared.exact]
    used_old = {i for i, _ in prepared.exact}
    used_new = {j for _, j in prepared.exact}
    candidates = []
    for row_index, row in enumerate(prepared.scores):
        for col_index in np.argsort(-row, kind="stable")[:candidates_per_clause]:
            score = float(row[col_index])
            if score >= threshold:
                candidates.append((score, prepared.old_indices[row_index],
                                   prepared.new_indices[int(col_index)]))
    candidates.sort(key=lambda item: (-item[0], item[1], item[2]))
    for score, i, j in candidates:
        if i not in used_old and j not in used_new:
            matches.append(ClauseMatch(prepared.old[i], prepared.new[j],
                                       round(score, 6), method))
            used_old.add(i)
            used_new.add(j)
    matches.sort(key=lambda match: match.old.start)
    return ClauseAlignment(
        tuple(matches),
        tuple(clause for i, clause in enumerate(prepared.old) if i not in used_old),
        tuple(clause for j, clause in enumerate(prepared.new) if j not in used_new),
    )


def align_policies(
    old: PolicyDocument,
    new: PolicyDocument,
    *,
    minimum_similarity: float | None = None,
    candidates_per_clause: int = 5,
    method: Literal["tfidf", "minilm"] = "tfidf",
    encoder=None,
) -> ClauseAlignment:
    """Align exact clauses, then assign distinct high-similarity candidates."""
    if old.role != "old" or new.role != "new":
        raise PolicyDeltaError("Expected old and new documents in that order.")
    if method not in ("tfidf", "minilm"):
        raise PolicyDeltaError("Alignment method must be tfidf or minilm.")
    threshold = alignment_threshold(method) if minimum_similarity is None else minimum_similarity
    if (isinstance(threshold, bool) or not isinstance(threshold, (int, float))
            or not math.isfinite(threshold) or not 0 < threshold <= 1):
        raise PolicyDeltaError("Minimum similarity must be between 0 and 1.")
    if type(candidates_per_clause) is not int or candidates_per_clause < 1:
        raise PolicyDeltaError("Candidate count must be a positive integer.")
    prepared = _prepare_alignment(old, new, method, encoder)
    return _assign_alignment(prepared, method, threshold, candidates_per_clause)


## Change detection

In [ ]:
@dataclass(frozen=True, slots=True)
class ChangeRecord:
    operation: Literal["UNCHANGED", "REPHRASED", "MODIFIED", "ADDED", "REMOVED", "UNCERTAIN"]
    old: PolicyClause | None
    new: PolicyClause | None
    signals: tuple[str, ...] = ()
    old_to_new: tuple[float, float, float] | None = None
    new_to_old: tuple[float, float, float] | None = None
    categories: tuple[str, ...] = ()


@lru_cache(maxsize=1)
def load_nli_model():
    """Load the NLI cross-encoder on first use."""
    try:
        from sentence_transformers import CrossEncoder
        return CrossEncoder("cross-encoder/nli-deberta-v3-small", max_length=512)
    except Exception as exc:
        raise PolicyDeltaError("Unable to load the NLI model.") from exc


_NUMBER_WORDS = dict(zip(
    "zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen fifteen sixteen seventeen eighteen nineteen twenty thirty forty fifty sixty seventy eighty ninety".split(),
    [*range(21), 30, 40, 50, 60, 70, 80, 90],
))
_NUMBER_PATTERN = re.compile(
    r"\b(\d+(?:\.\d+)?|" + "|".join(_NUMBER_WORDS) +
    r")\b\s*(days?|weeks?|months?|years?|hours?|minutes?|%|percent)?", re.I,
)
_NEGATION_PATTERN = re.compile(
    r"\b(?:not|never|no|without|cannot|can't|don't|doesn't|won't|isn't|aren't)\b", re.I
)
_MODAL_PATTERN = re.compile(
    r"\b(?:may|might|can|could|must|shall|will|should|only|unless|except|required|prohibited)\b", re.I
)


def _number_terms(text: str) -> tuple[tuple[str, str], ...]:
    terms = []
    for match in _NUMBER_PATTERN.finditer(text):
        value = match.group(1).lower()
        value = str(_NUMBER_WORDS.get(value, value))
        unit = (match.group(2) or "").lower().rstrip("s")
        terms.append((value, "%" if unit == "percent" else unit))
    return tuple(sorted(terms))


def change_signals(old: str, new: str) -> tuple[str, ...]:
    """Identify explicit linguistic differences requiring closer review."""
    signals = []
    for label, pattern in (("negation", _NEGATION_PATTERN),
                           ("modality", _MODAL_PATTERN)):
        if sorted(x.casefold() for x in pattern.findall(old)) != sorted(
                x.casefold() for x in pattern.findall(new)):
            signals.append(label)
    if _number_terms(old) != _number_terms(new):
        signals.append("quantity")
    return tuple(signals)


def _nli_probs(logits: np.ndarray) -> np.ndarray:
    if logits.ndim != 2 or logits.shape[1] != 3 or not np.isfinite(logits).all():
        raise PolicyDeltaError("NLI model returned invalid scores.")
    scores = np.exp(logits - logits.max(axis=1, keepdims=True))
    return scores / scores.sum(axis=1, keepdims=True)


def analyze_changes(
    alignment: ClauseAlignment, *, nli_model=None, confidence_threshold: float = 0.70,
) -> tuple[ChangeRecord, ...]:
    """Classify aligned clauses with bidirectional NLI and targeted checks."""
    if (isinstance(confidence_threshold, bool)
            or not isinstance(confidence_threshold, (float, int))
            or not math.isfinite(confidence_threshold)
            or not 0.5 < confidence_threshold < 1):
        raise PolicyDeltaError("NLI confidence threshold must be between 0.5 and 1.")

    findings: list[ChangeRecord] = []
    pending: list[tuple[ClauseMatch, tuple[str, ...]]] = []
    for match in alignment.matches:
        if match.old.normalized_text.casefold() == match.new.normalized_text.casefold():
            findings.append(ChangeRecord("UNCHANGED", match.old, match.new))
        else:
            pending.append((match, change_signals(match.old.text, match.new.text)))

    model = (nli_model if nli_model is not None else load_nli_model()) if pending else None
    pairs: list[tuple[str, str]] = []
    eligible: list[tuple[ClauseMatch, tuple[str, ...]]] = []
    for match, signals in pending:
        old, new = match.old.text, match.new.text
        too_long = len(old) > 1200 or len(new) > 1200 or len((old + new).split()) > 300
        if not too_long and getattr(model, "tokenizer", None) is not None:
            try:
                tokens = model.tokenizer(old, new, truncation=False)["input_ids"]
                too_long = len(tokens) > min(getattr(model, "max_seq_length", None) or 512, 512)
            except Exception as exc:
                raise PolicyDeltaError("Unable to validate NLI input length.") from exc
        if too_long:
            findings.append(ChangeRecord("UNCERTAIN", match.old, match.new,
                                         (*signals, "input_too_long")))
        else:
            eligible.append((match, signals))
            pairs.extend(((old, new), (new, old)))

    if pairs:
        try:
            logits = np.asarray(model.predict(pairs, batch_size=16,
                                               show_progress_bar=False), dtype=float)
        except Exception as exc:
            raise PolicyDeltaError("NLI inference failed.") from exc
        if logits.shape != (len(pairs), 3):
            raise PolicyDeltaError("NLI returned an unexpected number of predictions.")
        probabilities = _nli_probs(logits)
        for i, (match, signals) in enumerate(eligible):
            forward, reverse = probabilities[2 * i], probabilities[2 * i + 1]
            contradictory = max(forward[0], reverse[0]) >= confidence_threshold
            equivalent = min(forward[1], reverse[1]) >= confidence_threshold
            asymmetric = (forward[1] >= confidence_threshold) != (reverse[1] >= confidence_threshold)
            if signals or contradictory or asymmetric:
                operation = "MODIFIED"
            elif equivalent:
                operation = "REPHRASED"
            else:
                operation = "UNCERTAIN"
            findings.append(ChangeRecord(operation, match.old, match.new, signals,
                                         tuple(map(float, forward)), tuple(map(float, reverse))))

    findings.extend(ChangeRecord("REMOVED", clause, None)
                    for clause in alignment.unmatched_old)
    findings.extend(ChangeRecord("ADDED", None, clause)
                    for clause in alignment.unmatched_new)
    findings.sort(key=lambda finding: (
        finding.old.start if finding.old is not None else float("inf"),
        finding.new.start if finding.new is not None else float("inf"),
    ))
    return tuple(
        replace(
            finding,
            categories=classify_privacy_topics(
                *(clause.text for clause in (finding.old, finding.new)
                  if clause is not None)
            ),
        )
        for finding in findings
    )


## Privacy topics

In [ ]:
from dataclasses import replace


_TOPIC_RULES = {
    "data_collection": r"\b(?:collect\w*|gather\w*|obtain\w*|acquir\w*|capture\w*|receiv\w*)\b.{0,80}\b(?:data|information|details|identifiers?|addresses?|locations?|email|usage|device)\b|\b(?:data|information|details|identifiers?|locations?)\b.{0,80}\b(?:collect\w*|gather\w*|obtain\w*)\b",
    "data_use": r"\b(?:us(?:e|es|ed|ing)|process\w*|profil\w*|analy[sz]\w*|personaliz\w*)\b.{0,80}\b(?:data|information|activity|behavior|analytics|advertising|marketing|personalization|purposes?)\b|\b(?:data|information)\b.{0,70}\b(?:analytics|advertising|marketing|personalization)\b",
    "third_party_sharing": r"\b(?:shar\w*|disclos\w*|sell|sold|provid\w*)\b.{0,90}\b(?:data|information|details|advertisers?|partners?|third.part(?:y|ies)|affiliates?|vendors?|service providers?)\b|\b(?:third.part(?:y|ies)|advertisers?|partners?|affiliates?|vendors?|service providers?)\b.{0,90}\b(?:receiv\w*|access|disclos\w*|shar\w*)\b",
    "data_retention": r"\b(?:retain\w*|retention|stor(?:e|es|ed|ing|age)|keep|kept|delet\w*|eras\w*|remov\w*)\b.{0,80}\b(?:data|information|records?|accounts?|until|days?|months?|years?|hours?|period)\b|\b(?:data|information|records?|accounts?)\b.{0,80}\b(?:retain\w*|retention|stor(?:e|es|ed|ing|age)|delet\w*|eras\w*|kept)\b",
    "user_rights": r"\b(?:opt.?out|withdraw (?:your )?consent|data subject rights?|right to (?:access|erasure|deletion|correct|object|portability)|request (?:the )?(?:deletion|access|correction|copy)|privacy choices)\b|\b(?:you|users?|individuals?)\s+(?:may|can|have (?:the )?right to)\s+(?:request|access|delete|correct|withdraw|opt|object|view|erase|download|export)\b",
    "data_security": r"\b(?:encrypt\w*|security|secure|protect\w*|safeguard\w*|access controls?|data breaches?|authentication|unauthorized access)\b",
    "cookies_tracking": r"\b(?:cookies?|track(?:ing|ers?)|pixels?|web beacons?|advertising identifiers?|device identifiers?|session identifiers?)\b",
    "international_transfers": r"\b(?:international|cross.border|outside (?:the )?(?:eu|eea|country|region)|another country|other countries|foreign jurisdictions?)\b.{0,90}\b(?:data|information|transfer\w*|process\w*|stor\w*|servers?|countries|jurisdictions?)\b|\b(?:transfer\w*|process\w*|stor\w*)\b.{0,90}\b(?:outside (?:the )?(?:eu|eea|country|region)|overseas|other countries|foreign jurisdictions?|internationally)\b",
    "sensitive_data": r"\b(?:sensitive (?:personal )?(?:data|information)|health (?:data|information)|medical records?|biometric\w*|genetic\w*|precise (?:geolocation|location)|financial (?:data|information)|payment cards?|sexual orientation|racial|ethnic|religious beliefs?|criminal records?)\b",
    "children": r"\b(?:children|child|kids?|minors?|parental consent|under (?:the age of )?(?:13|16|18))\b",
    "policy_updates": r"\b(?:privacy policy|this policy)\b.{0,80}\b(?:updat\w*|chang\w*|modif\w*|effective|notif\w*|notice)\b|\b(?:notif\w*|updat\w*)\b.{0,80}\b(?:privacy policy|this policy)\b",
}
_TOPIC_PATTERNS = {
    category: re.compile(pattern, re.IGNORECASE)
    for category, pattern in _TOPIC_RULES.items()
}


def classify_privacy_topics(*passages: str) -> tuple[str, ...]:
    """Assign privacy topics supported by the supplied source passages."""
    if not passages or any(not isinstance(text, str) for text in passages):
        raise PolicyDeltaError("Privacy-topic classification requires text passages.")
    categories = tuple(
        category for category, pattern in _TOPIC_PATTERNS.items()
        if any(pattern.search(text) for text in passages)
    )
    return categories or ("other",)


## Evidence and significance

In [ ]:
@dataclass(frozen=True, slots=True)
class ChangeAssessment:
    finding: ChangeRecord
    significance: Literal["none", "low", "medium", "high", "uncertain"]
    explanation: str
    evidence: tuple[EvidenceSpan, ...]
    factors: tuple[str, ...]


_SHARING_RESTRICTION = re.compile(
    r"\b(?:do not|does not|will not|never|cannot|can't)\s+"
    r"(?:\w+\s+){0,2}(?:share|disclose|sell)\b", re.I,
)
_SHARING_PERMISSION = re.compile(
    r"\b(?:may|can|will)\s+(?:\w+\s+){0,2}"
    r"(?:share|disclose|sell)\b", re.I,
)


def assess_changes(
    findings: tuple[ChangeRecord, ...],
    old_document: PolicyDocument,
    new_document: PolicyDocument,
) -> tuple[ChangeAssessment, ...]:
    """Assess review priority using verified source passages and explicit rules."""
    if old_document.role != "old" or new_document.role != "new":
        raise PolicyDeltaError("Expected old and new documents in that order.")

    assessments = []
    for finding in findings:
        if finding.operation == "ADDED":
            valid = finding.old is None and finding.new is not None
        elif finding.operation == "REMOVED":
            valid = finding.old is not None and finding.new is None
        else:
            valid = finding.old is not None and finding.new is not None
        if not valid:
            raise PolicyDeltaError("Change operation does not match its source clauses.")

        evidence = []
        for clause, document in ((finding.old, old_document), (finding.new, new_document)):
            if clause is None:
                continue
            if clause.document_id != document.document_id:
                raise EvidenceError("Change refers to an unrelated document.")
            span = document.excerpt(clause.start, clause.end)
            if span.text != clause.text:
                raise EvidenceError("Change text does not match its evidence span.")
            verify_evidence(span, document)
            evidence.append(span)

        passages = [span.text for span in evidence]
        categories = classify_privacy_topics(*passages)
        topics = [category.replace("_", " ") for category in categories if category != "other"]
        subject = ", ".join(topics) if topics else "privacy-policy wording"
        old_text = finding.old.text if finding.old else ""
        new_text = finding.new.text if finding.new else ""
        signals = change_signals(old_text, new_text) if old_text and new_text else ()
        factors = list(signals)

        restriction = bool(_SHARING_RESTRICTION.search(old_text))
        permission = (bool(_SHARING_PERMISSION.search(new_text))
                      and not _NEGATION_PATTERN.search(new_text))
        sharing = "third_party_sharing" in categories
        relaxed = (finding.operation == "MODIFIED" and sharing
                   and restriction and permission
                   and not re.search(r"\b(?:unless|except)\b", old_text, re.I))
        removed_restriction = finding.operation == "REMOVED" and sharing and restriction

        if finding.operation == "UNCHANGED":
            significance = "none"
            explanation = "The corresponding source passages have no detected text change."
        elif finding.operation == "REPHRASED":
            significance = "low"
            explanation = ("Wording differs, but bidirectional inference supports "
                           "equivalent meaning. The source passages remain available for review.")
        elif finding.operation == "UNCERTAIN":
            significance = "uncertain"
            explanation = ("The available evidence is insufficient to determine whether "
                           "the corresponding passages changed in meaning.")
            if "input_too_long" in finding.signals:
                explanation = "The passages exceed the supported inference length and require review."
                factors.append("input_too_long")
        elif relaxed:
            significance = "high"
            factors.append("sharing_restriction_relaxed")
            explanation = ("The earlier passage explicitly restricts data sharing; "
                           "the updated passage permits it. Review the stated conditions and scope.")
        elif removed_restriction:
            significance = "high"
            factors.append("sharing_restriction_unmatched")
            explanation = ("An explicit restriction on sharing has no matched passage "
                           "in the updated document. Check whether it was moved or rephrased.")
        else:
            significance = "medium" if topics or signals else "low"
            if finding.operation == "ADDED":
                explanation = (f"A statement concerning {subject} has no match in the "
                               "earlier document. Check whether equivalent wording appears elsewhere.")
            elif finding.operation == "REMOVED":
                explanation = (f"A statement concerning {subject} has no match in the "
                               "updated document. Check whether it was moved or rephrased.")
            elif signals:
                explanation = (f"The matched statements concerning {subject} differ in "
                               f"{', '.join(signals)}. Review the source wording for its effect.")
            else:
                explanation = (f"Matched statements concerning {subject} differ in meaning "
                               "according to the comparison. Review both source passages.")

        assessments.append(ChangeAssessment(
            finding, significance, explanation, tuple(evidence), tuple(factors),
        ))
    return tuple(assessments)


## Comparison reports

In [ ]:
import csv
import io
from collections import Counter


_OPERATIONS = ("UNCHANGED", "REPHRASED", "MODIFIED", "ADDED", "REMOVED", "UNCERTAIN")
_SIGNIFICANCE = ("none", "low", "medium", "high", "uncertain")
_REPORT_COLUMNS = (
    "finding_id", "operation", "significance", "categories", "signals", "factors",
    "explanation", "old_document_id", "old_section", "old_start", "old_end",
    "old_text", "new_document_id", "new_section", "new_start", "new_end", "new_text",
)


def build_comparison_report(
    assessments: tuple[ChangeAssessment, ...],
    old_document: PolicyDocument,
    new_document: PolicyDocument,
) -> dict:
    """Create a deterministic report from source-verified assessments."""
    if not isinstance(assessments, tuple) or not all(
        isinstance(item, ChangeAssessment) for item in assessments
    ):
        raise PolicyDeltaError("Expected a tuple of change assessments.")
    for document in (old_document, new_document):
        if hashlib.sha256(document.text.encode("utf-8")).hexdigest() != document.sha256:
            raise EvidenceError("Source document integrity check failed.")
    checked = assess_changes(tuple(item.finding for item in assessments),
                             old_document, new_document)
    if checked != assessments:
        raise EvidenceError("An assessment differs from its verified source evidence.")

    rows = []
    for index, assessment in enumerate(assessments, start=1):
        finding = assessment.finding
        clauses = [clause for clause in (finding.old, finding.new) if clause is not None]
        categories = classify_privacy_topics(*(clause.text for clause in clauses))
        if finding.categories and finding.categories != categories:
            raise EvidenceError("Finding categories do not match the source passages.")

        def passage(clause):
            if clause is None:
                return None
            return {
                "document_id": clause.document_id,
                "section": clause.section,
                "start": clause.start,
                "end": clause.end,
                "text": clause.text,
            }

        rows.append({
            "finding_id": f"change-{index:04d}",
            "operation": finding.operation,
            "significance": assessment.significance,
            "categories": list(categories),
            "signals": list(finding.signals),
            "factors": list(assessment.factors),
            "explanation": assessment.explanation,
            "evidence": {"old": passage(finding.old), "new": passage(finding.new)},
            "nli": {"old_to_new": finding.old_to_new,
                    "new_to_old": finding.new_to_old},
        })

    operation_counts = Counter(row["operation"] for row in rows)
    significance_counts = Counter(row["significance"] for row in rows)
    topic_counts = Counter(topic for row in rows for topic in row["categories"])
    return {
        "documents": {
            role: {"document_id": doc.document_id, "sha256": doc.sha256,
                   "characters": len(doc.text)}
            for role, doc in (("old", old_document), ("new", new_document))
        },
        "summary": {
            "total_findings": len(rows),
            "review_findings": sum(row["operation"] in
                                   {"MODIFIED", "ADDED", "REMOVED", "UNCERTAIN"}
                                   for row in rows),
            "by_operation": {key: operation_counts[key] for key in _OPERATIONS},
            "by_significance": {key: significance_counts[key] for key in _SIGNIFICANCE},
            "by_topic": dict(sorted(topic_counts.items())),
        },
        "findings": rows,
    }


def report_json(report: dict) -> str:
    """Serialize a comparison without changing the original evidence text."""
    return json.dumps(report, ensure_ascii=False, indent=2)


def report_csv(report: dict) -> str:
    """Export one finding per row with spreadsheet-safe text cells."""
    output = io.StringIO(newline="")
    writer = csv.DictWriter(output, fieldnames=_REPORT_COLUMNS, lineterminator="\n")
    writer.writeheader()
    for finding in report["findings"]:
        old, new = finding["evidence"]["old"], finding["evidence"]["new"]
        row = {key: finding[key] for key in
               ("finding_id", "operation", "significance", "explanation")}
        row.update({key: "; ".join(finding[key]) for key in
                    ("categories", "signals", "factors")})
        for role, passage in (("old", old), ("new", new)):
            for key in ("document_id", "section", "start", "end", "text"):
                row[f"{role}_{key}"] = "" if passage is None else passage[key]
        for key, value in row.items():
            if isinstance(value, str) and re.match(r"^\s*[=+\-@]", value):
                row[key] = "'" + value
        writer.writerow(row)
    return output.getvalue()


def save_comparison_report(report: dict, directory: str | Path) -> tuple[Path, Path]:
    """Write JSON and CSV reports to a selected directory."""
    target = Path(directory)
    target.mkdir(parents=True, exist_ok=True)
    json_path, csv_path = target / "PolicyDelta_report.json", target / "PolicyDelta_findings.csv"
    json_path.write_text(report_json(report), encoding="utf-8")
    csv_path.write_text(report_csv(report), encoding="utf-8")
    return json_path, csv_path


## Interactive comparison

In [ ]:
import html
import gradio as gr


def read_policy_input(text: str | None, path: str | None, role: Literal["old", "new"]) -> PolicyDocument:
    """Load a pasted policy or one uploaded document."""
    if text and text.strip() and path:
        raise PolicyDeltaError("Use either pasted text or a file for each policy, not both.")
    if path:
        suffix = Path(path).suffix.lower()
        readers = {
            ".txt": PolicyDocument.from_txt,
            ".html": PolicyDocument.from_html_file,
            ".htm": PolicyDocument.from_html_file,
            ".pdf": PolicyDocument.from_pdf,
        }
        if suffix not in readers:
            raise PolicyDeltaError("Supported files: TXT, HTML, HTM, PDF.")
        return readers[suffix](path, role=role)
    return PolicyDocument.from_text(text or "", role=role)


def compare_policy_inputs(
    old_text: str | None, old_file: str | None,
    new_text: str | None, new_file: str | None,
    *, method: str = "minilm", minimum_similarity: float | None = None,
    confidence_threshold: float = 0.70, encoder=None, nli_model=None,
) -> dict:
    """Run the full comparison independently of the UI."""
    old = read_policy_input(old_text, old_file, "old")
    new = read_policy_input(new_text, new_file, "new")
    if max(len(segment_policy(old)), len(segment_policy(new))) > 300:
        raise PolicyDeltaError("This interface supports up to 300 clauses per policy.")
    alignment = align_policies(old, new, method=method,
                               minimum_similarity=minimum_similarity, encoder=encoder)
    findings = analyze_changes(alignment, nli_model=nli_model,
                               confidence_threshold=confidence_threshold)
    return build_comparison_report(assess_changes(findings, old, new), old, new)


def filter_report(report: dict | None, priority: str = "All", topic: str = "All",
                  operation: str = "All") -> list[dict]:
    """Select findings without changing the underlying comparison report."""
    if not report:
        return []
    results = [item for item in report["findings"]
               if (priority == "All" or item["significance"] == priority)
               and (topic == "All" or topic in item["categories"])
               and (operation == "All" or item["operation"] == operation)]
    rank = {"high": 0, "medium": 1, "uncertain": 2, "low": 3, "none": 4}
    return sorted(results, key=lambda item: rank[item["significance"]])


def finding_detail(report: dict | None, finding_id: str | None) -> str:
    """Render source evidence as escaped HTML."""
    if not report or not finding_id:
        return "<p>Select a finding to inspect its evidence.</p>"
    finding = next((item for item in report["findings"]
                    if item["finding_id"] == finding_id), None)
    if finding is None:
        return "<p>Finding not found.</p>"

    def block(role: str, title: str) -> str:
        passage = finding["evidence"][role]
        if passage is None:
            return f"<section><h4>{title}</h4><p>No corresponding passage.</p></section>"
        section = html.escape(passage["section"] or "Unspecified section")
        source = html.escape(passage["text"])
        return (f"<section><h4>{title}</h4>"
                f"<small>{section} · characters {passage['start']}–{passage['end']}</small>"
                f"<pre style='white-space:pre-wrap;overflow-wrap:anywhere'>{source}</pre>"
                "</section>")

    explanation = html.escape(finding["explanation"])
    categories = ", ".join(x.replace("_", " ") for x in finding["categories"])
    return (f"<p><strong>{html.escape(finding['operation'])}</strong> · "
            f"{html.escape(finding['significance'])} priority · "
            f"{html.escape(categories)}</p>"
            f"{block('old', 'Previous policy')}"
            f"{block('new', 'Updated policy')}"
            f"<p>{explanation}</p>")


def finding_view(report: dict | None, priority: str = "All", topic: str = "All",
                 operation: str = "All") -> tuple[list[list[str]], dict, str]:
    selected = filter_report(report, priority, topic, operation)
    table = [[item["finding_id"], item["operation"], item["significance"],
              ", ".join(item["categories"]), item["explanation"]]
             for item in selected]
    choices = [(f"{item['finding_id']} · {item['operation']} · {item['significance']}",
                item["finding_id"]) for item in selected]
    first = selected[0]["finding_id"] if selected else None
    return table, gr.update(choices=choices, value=first), finding_detail(report, first)


def run_comparison_ui(old_text, old_file, new_text, new_file, method,
                      similarity, confidence, priority="All", topic="All", operation="All"):
    try:
        report = compare_policy_inputs(old_text, old_file, new_text, new_file,
                                       method=method, minimum_similarity=similarity,
                                       confidence_threshold=confidence)
        summary = report["summary"]
        cutoff = alignment_threshold(method) if similarity is None else similarity
        cutoff_source = ("manual" if similarity is not None else
                         "calibrated" if method in CALIBRATED_MATCH_THRESHOLDS else "provisional")
        overview = (f"**{summary['total_findings']} aligned and unmatched findings** · "
                    f"**{summary['review_findings']} require review** · "
                    f"**{summary['by_significance']['high']} high-priority** · "
                    f"Matching cutoff: **{cutoff:.3f} ({cutoff_source})**")
        rows, options, detail = finding_view(report, priority, topic, operation)
        directory = Path(tempfile.mkdtemp(prefix="policydelta-"))
        json_path, csv_path = save_comparison_report(report, directory)
        return report, overview, rows, options, detail, str(json_path), str(csv_path)
    except PolicyDeltaError as exc:
        return (None, f"**Unable to compare:** {exc}", [],
                gr.update(choices=[], value=None), "", None, None)


def build_policy_delta_app() -> gr.Blocks:
    """Create the interactive policy comparison interface."""
    with gr.Blocks(title="PolicyDelta") as app:
        gr.Markdown("# PolicyDelta\nSemantic Privacy Policy Change Detector")
        gr.Markdown("Compare policy revisions and inspect supporting text. "
                    "Findings are screening results, not legal determinations. "
                    "Colab may expose a public Gradio link; use non-confidential documents.")
        with gr.Row():
            with gr.Column():
                old_text = gr.Textbox(lines=7, label="Previous policy text")
                old_file = gr.File(label="Or upload previous policy",
                                   file_types=[".txt", ".html", ".htm", ".pdf"],
                                   type="filepath")
            with gr.Column():
                new_text = gr.Textbox(lines=7, label="Updated policy text")
                new_file = gr.File(label="Or upload updated policy",
                                   file_types=[".txt", ".html", ".htm", ".pdf"],
                                   type="filepath")
        with gr.Accordion("Comparison settings", open=False):
            method = gr.Radio(["minilm", "tfidf"], value="minilm", label="Matching method")
            similarity = gr.Number(value=None, precision=4,
                                   label="Matching threshold override (leave empty for automatic)")
            gr.Markdown("Automatic uses a labeled-data calibration when loaded; "
                        "otherwise a provisional heuristic. NLI confidence is not calibrated.")
            confidence = gr.Slider(0.55, 0.95, value=0.70, step=0.05,
                                   label="NLI confidence threshold")
        with gr.Row():
            compare = gr.Button("Compare policies", variant="primary")
            example = gr.Button("Load sample")
        summary = gr.Markdown("Upload or paste two policies to begin.")
        report_state = gr.State(None)
        with gr.Row():
            priority = gr.Dropdown(["All", *_SIGNIFICANCE], value="All", label="Priority")
            topic = gr.Dropdown(["All", *_TOPIC_RULES, "other"], value="All", label="Topic")
            operation = gr.Dropdown(["All", *_OPERATIONS], value="All", label="Change type")
        table = gr.Dataframe(headers=["ID", "Change", "Priority", "Topics", "Explanation"],
                             datatype="str", interactive=False, wrap=True, label="Findings")
        selection = gr.Dropdown(choices=[], label="Inspect finding")
        detail = gr.HTML("<p>Select a finding to inspect its evidence.</p>")
        with gr.Row():
            json_output = gr.File(label="JSON report")
            csv_output = gr.File(label="CSV findings")

        compare.click(run_comparison_ui,
                      inputs=[old_text, old_file, new_text, new_file, method,
                              similarity, confidence, priority, topic, operation],
                      outputs=[report_state, summary, table, selection, detail,
                               json_output, csv_output])

        for control in (priority, topic, operation):
            control.change(finding_view,
                           inputs=[report_state, priority, topic, operation],
                           outputs=[table, selection, detail])
        selection.change(finding_detail, inputs=[report_state, selection], outputs=detail)
        example.click(
            lambda: ("We do not share personal information with advertisers.\n\n"
                     "We retain account records for 30 days.",
                     "We may share personal information with advertisers.\n\n"
                     "We retain account records for 90 days.", None, None),
            outputs=[old_text, new_text, old_file, new_file],
        )
    return app


## Alignment evaluation


In [ ]:
import csv


@dataclass(frozen=True, slots=True)
class LabeledPolicyPair:
    site: str
    pair_id: str
    split: Literal["calibration", "test"]
    old: PolicyDocument
    new: PolicyDocument
    gold_matches: frozenset[tuple[int, int]]
    source: str


def load_alignment_benchmark(manifest: str | Path) -> list[LabeledPolicyPair]:
    """Load reviewed clause correspondences from a source-attributed CSV manifest."""
    path = Path(manifest).resolve()
    root = path.parent
    required = {"site", "pair_id", "split", "old_path", "new_path", "gold_matches", "source"}
    try:
        with path.open(encoding="utf-8-sig", newline="") as handle:
            rows = list(csv.DictReader(handle))
    except (OSError, UnicodeError) as exc:
        raise PolicyDeltaError("Unable to read benchmark manifest.") from exc
    if not rows or not required.issubset(rows[0]):
        raise PolicyDeltaError("Benchmark manifest is empty or missing required columns.")

    def document(relative: str, role: Literal["old", "new"]) -> PolicyDocument:
        if not isinstance(relative, str) or not relative.strip():
            raise PolicyDeltaError("Benchmark document path is missing.")
        file = (root / relative).resolve()
        if not file.is_relative_to(root) or not file.is_file():
            raise PolicyDeltaError("Benchmark source file is missing or outside the manifest directory.")
        readers = {".txt": PolicyDocument.from_txt, ".html": PolicyDocument.from_html_file,
                   ".htm": PolicyDocument.from_html_file, ".pdf": PolicyDocument.from_pdf}
        if file.suffix.lower() == ".md":
            try:
                return PolicyDocument.from_text(file.read_text(encoding="utf-8-sig"), role=role)
            except (OSError, UnicodeError) as exc:
                raise PolicyDeltaError("Unable to read benchmark Markdown source.") from exc
        if file.suffix.lower() not in readers:
            raise PolicyDeltaError("Benchmark documents must be TXT, HTML, Markdown, or PDF.")
        return readers[file.suffix.lower()](file, role=role)

    pairs = []
    seen, sites = set(), {}
    file_hashes: dict[str, str] = {}
    for row in rows:
        site, pair_id, split = ((row[field] or "").strip()
                                for field in ("site", "pair_id", "split"))
        site = site.casefold().removeprefix("www.")
        if (not site or not pair_id or not (row["source"] or "").strip()
                or split not in ("calibration", "test") or (site, pair_id) in seen):
            raise PolicyDeltaError("Invalid or repeated benchmark identity, source, or split.")
        if site in sites and sites[site] != split:
            raise PolicyDeltaError("The same site cannot occur in calibration and test splits.")
        seen.add((site, pair_id))
        sites[site] = split
        old = document(row["old_path"], "old")
        new = document(row["new_path"], "new")
        for policy in (old, new):
            if policy.sha256 in file_hashes and file_hashes[policy.sha256] != split:
                raise PolicyDeltaError("Policy content appears in both benchmark splits.")
            file_hashes[policy.sha256] = split
        try:
            raw = json.loads(row["gold_matches"])
            if not isinstance(raw, list):
                raise ValueError("Expected list")
            gold = []
            for entry in raw:
                if (not isinstance(entry, list) or len(entry) != 2
                        or any(type(index) is not int or index < 0 for index in entry)):
                    raise ValueError("Expected nonnegative clause index pairs")
                gold.append(tuple(entry))
        except (ValueError, TypeError, json.JSONDecodeError) as exc:
            raise PolicyDeltaError("gold_matches must be JSON clause index pairs.") from exc
        old_count, new_count = len(segment_policy(old)), len(segment_policy(new))
        if (any(i >= old_count or j >= new_count for i, j in gold)
                or len(set(gold)) != len(gold)
                or len({i for i, _ in gold}) != len(gold)
                or len({j for _, j in gold}) != len(gold)):
            raise PolicyDeltaError("Gold matches must be unique, one-to-one, and within clause bounds.")
        pairs.append(LabeledPolicyPair(site, pair_id, split, old, new, frozenset(gold),
                                       row["source"].strip()))
    if set(sites.values()) != {"calibration", "test"}:
        raise PolicyDeltaError("Benchmark requires independent calibration and test sites.")
    return pairs


def _alignment_counts(prepared: _PreparedAlignment, alignment: ClauseAlignment,
                      gold: frozenset[tuple[int, int]]) -> tuple[int, int, int]:
    old_index = {clause.clause_id: i for i, clause in enumerate(prepared.old)}
    new_index = {clause.clause_id: j for j, clause in enumerate(prepared.new)}
    predicted = {(old_index[match.old.clause_id], new_index[match.new.clause_id])
                 for match in alignment.matches}
    return len(predicted & gold), len(predicted - gold), len(gold - predicted)


def _alignment_metrics(counts: tuple[int, int, int]) -> dict[str, float | int]:
    tp, fp, fn = counts
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {"precision": round(precision, 4), "recall": round(recall, 4),
            "f1": round(2 * tp / (2 * tp + fp + fn), 4) if 2 * tp + fp + fn else 0.0,
            "tp": tp, "fp": fp, "fn": fn}


def calibrate_alignment_threshold(
    pairs: list[LabeledPolicyPair], *, method: Literal["tfidf", "minilm"] = "tfidf", encoder=None,
) -> dict:
    """Choose a matching cutoff on calibration sites only, optimizing micro-F1."""
    if method not in ("tfidf", "minilm"):
        raise PolicyDeltaError("Unsupported alignment method.")
    selected = [pair for pair in pairs if pair.split == "calibration"]
    if not selected or not any(pair.gold_matches for pair in selected):
        raise PolicyDeltaError("Calibration requires labeled positive correspondences.")
    prepared = [(_prepare_alignment(p.old, p.new, method, encoder), p.gold_matches)
                for p in selected]
    candidates = sorted({float(score) for item, _ in prepared for score in item.scores.flat
                         if np.isfinite(score) and 0 < score <= 1} | {1.0})
    if len(candidates) > 101:
        positions = np.linspace(0, len(candidates) - 1, 101, dtype=int)
        candidates = sorted({candidates[int(i)] for i in positions})
    best = None
    for cutoff in candidates:
        totals = tuple(sum(values) for values in zip(*(
            _alignment_counts(item, _assign_alignment(item, method, cutoff, 5), gold)
            for item, gold in prepared
        )))
        metrics = _alignment_metrics(totals)
        tp, fp, fn = totals
        objective = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
        precision = tp / (tp + fp) if tp + fp else 0.0
        ranking = (objective, precision, cutoff)
        if best is None or ranking > best[0]:
            best = (ranking, metrics)
    return {"method": method, "threshold": best[0][2], "calibration_metrics": best[1],
            "calibration_sites": len({p.site for p in selected}),
            "calibration_pairs": len(selected)}


def evaluate_alignment_threshold(
    pairs: list[LabeledPolicyPair], calibration: dict, *, encoder=None,
) -> dict:
    """Evaluate a fixed calibrated cutoff on held-out sites."""
    method, threshold = calibration["method"], calibration["threshold"]
    if (method not in ("tfidf", "minilm") or type(threshold) not in (float, int)
            or not math.isfinite(threshold) or not 0 < threshold <= 1):
        raise PolicyDeltaError("Invalid alignment calibration.")
    selected = [pair for pair in pairs if pair.split == "test"]
    if not selected:
        raise PolicyDeltaError("No held-out test data was provided.")
    totals = [0, 0, 0]
    for pair in selected:
        prepared = _prepare_alignment(pair.old, pair.new, method, encoder)
        counts = _alignment_counts(prepared, _assign_alignment(prepared, method, threshold, 5),
                                   pair.gold_matches)
        totals = [a + b for a, b in zip(totals, counts)]
    return {"method": method, "threshold": threshold,
            "test_metrics": _alignment_metrics(tuple(totals)),
            "test_sites": len({p.site for p in selected}), "test_pairs": len(selected)}


def use_alignment_calibration(calibration: dict) -> None:
    """Activate a validated calibration for subsequent comparisons."""
    method = calibration.get("method")
    threshold = calibration.get("threshold")
    if (method not in ("tfidf", "minilm") or type(threshold) not in (float, int)
            or not math.isfinite(threshold) or not 0 < threshold <= 1
            or not calibration.get("calibration_pairs") or not calibration.get("calibration_sites")):
        raise PolicyDeltaError("Invalid alignment calibration.")
    CALIBRATED_MATCH_THRESHOLDS[method] = float(threshold)


## Historical privacy policies

In [ ]:
import csv
import io
import json
import hashlib
import re
from pathlib import Path
from urllib.parse import quote

import requests

OTA_REPOSITORY = "OpenTermsArchive/contrib-versions"
OTA_API = f"https://api.github.com/repos/{OTA_REPOSITORY}"
OTA_PROVIDERS = (
    "Airbnb", "Apple", "Atlassian", "Cloudflare", "Coinbase", "Coursera",
    "Discord", "Dropbox", "Evernote", "Facebook", "Reddit", "Spotify",
)
MANIFEST_COLUMNS = ("site", "pair_id", "split", "old_path", "new_path", "gold_matches", "source")


def _github_json(url: str, session: requests.Session) -> dict | list:
    try:
        response = session.get(url, timeout=30, headers={"Accept": "application/vnd.github+json"})
        if response.status_code == 403:
            raise PolicyDeltaError("GitHub API access was limited; retry later or supply an authenticated session.")
        response.raise_for_status()
        return response.json()
    except (requests.RequestException, ValueError) as exc:
        raise PolicyDeltaError(f"GitHub metadata could not be retrieved: {url}") from exc


def _download_version(entry: dict, session: requests.Session) -> str:
    url = entry.get("download_url", "")
    if not url.startswith(f"https://raw.githubusercontent.com/{OTA_REPOSITORY}/"):
        raise PolicyDeltaError("Unexpected source URL in historical dataset.")
    if not 0 < entry.get("size", 0) <= SETTINGS.max_document_chars:
        raise PolicyDeltaError("Source policy exceeds the supported document size.")
    try:
        response = session.get(url, timeout=45)
        response.raise_for_status()
        raw = response.content
        if len(raw) > SETTINGS.max_document_chars * 4:
            raise PolicyDeltaError("Downloaded policy is too large.")
        git_blob = hashlib.sha1(f"blob {len(raw)}\0".encode() + raw).hexdigest()
        if git_blob != entry["sha"]:
            raise PolicyDeltaError("Downloaded policy does not match its Git source hash.")
        return raw.decode("utf-8-sig")
    except (requests.RequestException, UnicodeError, KeyError) as exc:
        raise PolicyDeltaError("Unable to download a verified UTF-8 policy version.") from exc


def _version_candidates(versions: list[dict]) -> list[tuple[dict, dict]]:
    eligible = [item for item in versions
                if item.get("type") == "file" and re.fullmatch(r"\d{4}-\d{2}-\d{2}T[^/]+\.md", item["name"])
                and 500 <= item.get("size", 0) <= SETTINGS.max_document_chars]
    eligible.sort(key=lambda entry: entry["name"])
    return [(eligible[i - 1], eligible[i]) for i in range(len(eligible) - 1, 0, -1)
            if eligible[i - 1].get("sha") != eligible[i].get("sha")][:6]


def _save_historical_pairs(records: list[dict], directory: Path, *, revision: str) -> dict:
    if len({record["site"] for record in records}) < 2:
        raise PolicyDeltaError("At least two independent sites are needed for a split.")
    directory.mkdir(parents=True, exist_ok=True)
    (directory / "policies").mkdir(exist_ok=True)
    sites = sorted({record["site"] for record in records},
                   key=lambda site: hashlib.sha256(f"{SETTINGS.seed}:{site}".encode()).digest())
    calibration = set(sites[:max(1, min(len(sites) - 1, round(len(sites) * 0.7)))])
    draft, review, suggestions, diagnostics, provenance = [], [], [], [], []
    for record in records:
        site = record["site"]
        pair_id = record["pair_id"]
        old_path, new_path = (Path("policies") / f"{pair_id}-{role}.md" for role in ("old", "new"))
        for path, role in ((old_path, "old"), (new_path, "new")):
            with (directory / path).open("w", encoding="utf-8", newline="") as handle:
                handle.write(record[role])
        old = PolicyDocument.from_text(record["old"], role="old")
        new = PolicyDocument.from_text(record["new"], role="new")
        old_clauses, new_clauses = segment_policy(old), segment_policy(new)
        alignment = align_policies(old, new, method="tfidf", minimum_similarity=0.4)
        scores = tfidf_similarity(old_clauses, new_clauses)
        for clause in [*old_clauses, *new_clauses]:
            source = old if clause.document_id == old.document_id else new
            if source.text[clause.start:clause.end] != clause.text:
                raise EvidenceError("Historical policy has an invalid evidence span.")
        for i, old_clause in enumerate(old_clauses):
            review.append({"pair_id": pair_id, "old_index": i, "matched_new_index": "", "reviewed": ""})
            for j in scores[i].argsort(kind="stable")[::-1][:3]:
                new_clause = new_clauses[int(j)]
                suggestions.append({"pair_id": pair_id, "old_index": i, "new_index": int(j),
                                    "similarity": round(float(scores[i, j]), 5),
                                    "old_text": old_clause.text, "new_text": new_clause.text})
        split = "calibration" if site in calibration else "test"
        source = f"https://github.com/{OTA_REPOSITORY}/tree/{revision}"
        draft.append({"site": site, "pair_id": pair_id, "split": split,
                      "old_path": str(old_path), "new_path": str(new_path),
                      "gold_matches": "", "source": source})
        diagnostics.append({"site": site, "pair_id": pair_id, "split": split,
                            "old_clauses": len(old_clauses), "new_clauses": len(new_clauses),
                            "matched": len(alignment.matches),
                            "unmatched_old": len(alignment.unmatched_old),
                            "unmatched_new": len(alignment.unmatched_new),
                            "source_offsets_valid": True})
        provenance.append({"site": site, "pair_id": pair_id, "revision": revision,
                           "old_url": record["old_url"], "new_url": record["new_url"],
                           "old_sha256": old.sha256, "new_sha256": new.sha256,
                           "old_git_blob": record.get("old_git_blob"),
                           "new_git_blob": record.get("new_git_blob"),
                           "license": "Open Terms Archive contributors; verify the repository LICENSE before redistribution"})

    def write_csv(filename: str, rows: list[dict], fields: list[str] | tuple[str, ...]) -> None:
        with (directory / filename).open("w", encoding="utf-8", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=fields)
            writer.writeheader()
            writer.writerows(rows)

    write_csv("manifest_draft.csv", draft, MANIFEST_COLUMNS)
    write_csv("review_decisions.csv", review, ("pair_id", "old_index", "matched_new_index", "reviewed"))
    write_csv("review_suggestions.csv", suggestions,
              ("pair_id", "old_index", "new_index", "similarity", "old_text", "new_text"))
    write_csv("extraction_results.csv", diagnostics, tuple(diagnostics[0]))
    (directory / "provenance.json").write_text(json.dumps(provenance, indent=2), encoding="utf-8")
    return {"pair_count": len(records), "site_count": len(sites), "calibration_sites": len(calibration),
            "test_sites": len(sites) - len(calibration), "review_rows": len(review),
            "dataset_path": str(directory), "source_revision": revision}


def prepare_historical_policies(
    directory: str | Path = "/content/benchmark", *,
    providers: tuple[str, ...] = OTA_PROVIDERS,
    max_pairs: int = 8,
    revision: str | None = None,
    session: requests.Session | None = None,
) -> dict:
    """Retrieve verifiable historical privacy-policy pairs and prepare human review files."""
    if not 2 <= max_pairs <= 20:
        raise PolicyDeltaError("Pair limit must be between 2 and 20.")
    session = session or requests.Session()
    if revision is None:
        revision = _github_json(f"{OTA_API}/commits/main", session)["sha"]
    if not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise PolicyDeltaError("A full Git commit hash is required.")
    records = []
    failures = []
    root = _github_json(f"{OTA_API}/contents?ref={revision}", session)
    if not isinstance(root, list):
        raise PolicyDeltaError("Unexpected repository directory listing.")
    available = {item["name"].casefold(): item["name"] for item in root
                 if item.get("type") == "dir"}
    selected = [available[name.casefold()] for name in providers
                if name.casefold() in available]
    if len(selected) < 2:
        selected = [item["name"] for item in root if item.get("type") == "dir"]
    for provider in selected:
        if len(records) >= max_pairs:
            break
        try:
            items = _github_json(f"{OTA_API}/contents/{quote(provider, safe='')}?ref={revision}", session)
            if not isinstance(items, list):
                continue
            options = sorted((item for item in items
                              if "privacy" in item.get("name", "").casefold()),
                             key=lambda item: (item["name"].casefold() != "privacy policy", item["name"]))
            for option in options:
                path = option["path"]
                if option["type"] == "dir":
                    versions = _github_json(f"{OTA_API}/contents/{quote(path, safe='/')}?ref={revision}", session)
                    candidates = _version_candidates(versions)
                elif option["type"] == "file" and path.lower().endswith(".md"):
                    commits = _github_json(
                        f"{OTA_API}/commits?path={quote(path, safe='/')}&sha={revision}&per_page=20", session)
                    candidates = []
                    entries = []
                    for commit in commits:
                        sha = commit["sha"]
                        try:
                            entry = _github_json(f"{OTA_API}/contents/{quote(path, safe='/')}?ref={sha}", session)
                        except PolicyDeltaError:
                            continue
                        if (entry.get("type") == "file" and
                                500 <= entry.get("size", 0) <= SETTINGS.max_document_chars):
                            entry = dict(entry, name=f"{commit['commit']['committer']['date'][:10]}-{sha[:10]}.md")
                            entries.append(entry)
                    candidates = [(entries[i], entries[i - 1]) for i in range(1, len(entries))
                                  if entries[i]["sha"] != entries[i - 1]["sha"]]
                else:
                    continue
                for older, newer in candidates[:6]:
                    try:
                        old, new = _download_version(older, session), _download_version(newer, session)
                        if old == new or min(len(old), len(new)) < 500:
                            continue
                        old_doc = PolicyDocument.from_text(old, role="old")
                        new_doc = PolicyDocument.from_text(new, role="new")
                        if min(len(segment_policy(old_doc)), len(segment_policy(new_doc))) < 5:
                            continue
                    except PolicyDeltaError:
                        continue
                    slug = re.sub(r"[^a-z0-9]+", "-", provider.casefold()).strip("-")
                    pair_id = f"{slug}-{older['name'][:10]}-{newer['name'][:10]}"
                    records.append({"site": slug, "pair_id": pair_id, "old": old, "new": new,
                                    "old_url": older["download_url"], "new_url": newer["download_url"],
                                    "old_git_blob": older["sha"], "new_git_blob": newer["sha"]})
                    break
                if records and records[-1]["site"] == re.sub(r"[^a-z0-9]+", "-", provider.casefold()).strip("-"):
                    break
        except (PolicyDeltaError, KeyError, TypeError) as exc:
            failures.append(f"{provider}: {exc}")
    if len(records) < 2:
        reason = failures[-1] if failures else "No eligible privacy-policy histories were found."
        raise PolicyDeltaError(f"Insufficient historical policies retrieved ({len(records)}). {reason}")
    return _save_historical_pairs(records, Path(directory), revision=revision)


def finalize_reviewed_manifest(directory: str | Path) -> Path:
    """Publish an evaluable manifest only when every source clause is reviewed."""
    root = Path(directory)
    with (root / "manifest_draft.csv").open(encoding="utf-8", newline="") as stream:
        draft = list(csv.DictReader(stream))
    with (root / "review_decisions.csv").open(encoding="utf-8", newline="") as stream:
        decisions = list(csv.DictReader(stream))
    provenance = {entry["pair_id"]: entry for entry in
                  json.loads((root / "provenance.json").read_text(encoding="utf-8"))}
    indexed: dict[str, list[dict]] = {}
    for row in decisions:
        indexed.setdefault(row["pair_id"], []).append(row)
    if set(indexed) != {row["pair_id"] for row in draft}:
        raise PolicyDeltaError("Review decisions do not cover the benchmark pairs.")
    for row in draft:
        decisions_for_pair = indexed[row["pair_id"]]
        old = PolicyDocument.from_text((root / row["old_path"]).read_text(encoding="utf-8"), role="old")
        new = PolicyDocument.from_text((root / row["new_path"]).read_text(encoding="utf-8"), role="new")
        source = provenance.get(row["pair_id"])
        if (not source or old.sha256 != source["old_sha256"]
                or new.sha256 != source["new_sha256"]):
            raise EvidenceError("Historical policy content changed after extraction.")
        old_count, new_count = len(segment_policy(old)), len(segment_policy(new))
        if len(decisions_for_pair) != old_count:
            raise PolicyDeltaError("Every old clause must have exactly one review decision.")
        seen_old, seen_new, gold = set(), set(), []
        for decision in decisions_for_pair:
            if decision["reviewed"].strip() != "1":
                raise PolicyDeltaError("Review decisions are incomplete; no gold labels have been published.")
            try:
                i = int(decision["old_index"])
                if i in seen_old or i < 0 or i >= old_count:
                    raise ValueError
                seen_old.add(i)
                j_text = decision["matched_new_index"].strip()
                if j_text:
                    j = int(j_text)
                    if j < 0 or j >= new_count or j in seen_new:
                        raise ValueError
                    seen_new.add(j)
                    gold.append([i, j])
            except (ValueError, TypeError) as exc:
                raise PolicyDeltaError("A review decision contains invalid clause indices.") from exc
        row["gold_matches"] = json.dumps(sorted(gold))
    output = root / "manifest.csv"
    candidate = root / "manifest.candidate.csv"
    try:
        with candidate.open("w", encoding="utf-8", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=MANIFEST_COLUMNS)
            writer.writeheader()
            writer.writerows(draft)
        load_alignment_benchmark(candidate)
        candidate.replace(output)
    finally:
        candidate.unlink(missing_ok=True)
    return output


In [ ]:
if RUNTIME["environment"] == "colab":
    try:
        HISTORICAL_DATA = prepare_historical_policies("/content/benchmark")
        print(json.dumps(HISTORICAL_DATA, indent=2))
        from shutil import make_archive
        DATASET_ARCHIVE = make_archive("/content/PolicyDelta_Historical_Policies", "zip", "/content/benchmark")
        print(f"Prepared dataset archive: {DATASET_ARCHIVE}")
        print("Review labels before calling load_alignment_benchmark().")
    except PolicyDeltaError as exc:
        print(f"Historical dataset could not be prepared: {exc}")
else:
    print("Run prepare_historical_policies() in Colab to retrieve the historical archive.")

## Validation tests

In [ ]:
class PolicyDeltaTests(unittest.TestCase):
    def setUp(self) -> None:
        self.document = PolicyDocument.from_text(
            "We retain personal data for 30 days.", role="old", document_id="sample"
        )

    def test_default_settings(self):
        self.assertEqual(Settings().seed, 42)

    def test_invalid_settings(self):
        for values in ({"seed": -1}, {"seed": True}, {"max_document_chars": 0},
                       {"max_pdf_pages": 0}, {"max_pdf_bytes": 0}):
            with self.subTest(values=values), self.assertRaises(PolicyDeltaError):
                Settings(**values)

    def test_document_checksum(self):
        expected = hashlib.sha256(self.document.text.encode("utf-8")).hexdigest()
        self.assertEqual(self.document.sha256, expected)

    def test_empty_document(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_text("  ", role="old", document_id="empty")

    def test_input_limit(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_text("long text", role="old", document_id="large",
                                     settings=Settings(max_document_chars=5))

    def test_invalid_role(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_text("Valid text", role="invalid", document_id="bad")

    def test_evidence_exact_match(self):
        start = self.document.text.index("30 days")
        span = self.document.excerpt(start, start + len("30 days"))
        verify_evidence(span, self.document)
        self.assertEqual(span.text, "30 days")

    def test_tampered_evidence(self):
        span = EvidenceSpan("sample", 0, 2, "No")
        with self.assertRaises(EvidenceError):
            verify_evidence(span, self.document)

    def test_evidence_other_document(self):
        span = EvidenceSpan("another", 0, 2, "We")
        with self.assertRaises(EvidenceError):
            verify_evidence(span, self.document)

    def test_out_of_bounds(self):
        with self.assertRaises(EvidenceError):
            self.document.excerpt(0, len(self.document.text) + 1)

    def test_unicode_offsets(self):
        doc = PolicyDocument.from_text("Adatkezelés ✓", role="new", document_id="unicode")
        span = doc.excerpt(12, 13)
        self.assertEqual(span.text, "✓")
        verify_evidence(span, doc)

    def test_json_serialization(self):
        result = json.loads(document_json(self.document))
        self.assertEqual(result["role"], "old")
        self.assertEqual(result["sha256"], self.document.sha256)

    def test_generated_document_id(self):
        first = PolicyDocument.from_text("Policy text", role="old")
        second = PolicyDocument.from_text("Policy text", role="old")
        self.assertEqual(first.document_id, second.document_id)
        self.assertNotEqual(first.document_id,
                            PolicyDocument.from_text("Policy text", role="new").document_id)

    def test_txt_ingestion_preserves_offsets(self):
        content = "Data retention: 30 days.\r\nPrivacy ✓"
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.TXT"
            path.write_bytes(content.encode("utf-8"))
            document = PolicyDocument.from_txt(path, role="new")
        self.assertEqual(document.text, content)
        start = document.text.index("Privacy")
        verify_evidence(document.excerpt(start, len(document.text)), document)

    def test_txt_utf8_bom(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.txt"
            path.write_bytes("Consent required".encode("utf-8-sig"))
            document = PolicyDocument.from_txt(str(path), role="old")
        self.assertEqual(document.text, "Consent required")

    def test_txt_invalid_encoding(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.txt"
            path.write_bytes(b"\xff\xfe")
            with self.assertRaises(PolicyDeltaError):
                PolicyDocument.from_txt(path, role="old")

    def test_txt_missing_or_unsupported_file(self):
        with tempfile.TemporaryDirectory() as directory:
            for name in ("missing.txt", "policy.pdf"):
                with self.subTest(name=name), self.assertRaises(PolicyDeltaError):
                    PolicyDocument.from_txt(Path(directory) / name, role="old")

    def test_txt_character_limit(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.txt"
            path.write_text("123456", encoding="utf-8")
            with self.assertRaises(PolicyDeltaError):
                PolicyDocument.from_txt(path, role="new",
                                        settings=Settings(max_document_chars=5))

    def test_txt_rejects_invalid_path(self):
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_txt(None, role="old")

    def test_html_extraction_and_evidence(self):
        markup = ("<html><head><title>Website</title><script>tracker()</script></head>"
                  "<body><nav>Sign in</nav><main><h1>Privacy Policy</h1>"
                  "<p>We <strong>do not</strong> share personal data.</p>"
                  "<p>Records are kept for 30 days.</p></main>"
                  "<footer>Contact us</footer></body></html>")
        document = PolicyDocument.from_html(markup, role="new")
        self.assertEqual(document.text, "Privacy Policy\n\nWe do not share personal data."
                                          "\n\nRecords are kept for 30 days.")
        start = document.text.index("do not")
        verify_evidence(document.excerpt(start, start + 6), document)

    def test_html_without_semantic_main(self):
        markup = ("<div class='navigation'>Ignore navigation</div>"
                  "<div class='policy'><div>Collected data includes email.</div>"
                  "<div>We delete it on request.</div></div>")
        document = PolicyDocument.from_html(markup, role="old")
        self.assertEqual(document.text, "Collected data includes email.\n\n"
                                        "We delete it on request.")

    def test_html_lists_and_entities(self):
        markup = "<main><h2>Your rights</h2><ul><li>Access &amp; correction</li>" \
                 "<li>Delete your data</li></ul></main>"
        document = PolicyDocument.from_html(markup, role="old")
        self.assertEqual(document.text, "Your rights\n\nAccess & correction\n\n"
                                        "Delete your data")

    def test_html_nested_lists_and_line_breaks(self):
        markup = ("<main><ul><li>Data requests<ul><li>Deletion</li></ul></li></ul>"
                  "<p>Contact us<br>for assistance</p>"
                  "<p style='display:none'>Hidden text</p></main>")
        doc = PolicyDocument.from_html(markup, role="new")
        self.assertEqual(doc.text, "Data requests\n\nDeletion\n\nContact us\n\n"
                                   "for assistance")

    def test_html_file_ingestion(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "privacy.HTM"
            path.write_bytes("<main><p>Privacy ✓</p></main>".encode("utf-8-sig"))
            document = PolicyDocument.from_html_file(path, role="new")
        self.assertEqual(document.text, "Privacy ✓")

    def test_html_malformed_and_empty(self):
        document = PolicyDocument.from_html("<main><p>Data is retained", role="new")
        self.assertEqual(document.text, "Data is retained")
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_html("<script>no policy text</script>", role="old")

    def test_html_invalid_files(self):
        with tempfile.TemporaryDirectory() as directory:
            missing = Path(directory) / "missing.html"
            wrong = Path(directory) / "policy.txt"
            bad = Path(directory) / "invalid.html"
            bad.write_bytes(b"\xff\xfe")
            for path in (missing, wrong, bad):
                with self.subTest(path=path.name), self.assertRaises(PolicyDeltaError):
                    PolicyDocument.from_html_file(path, role="old")

    def test_html_size_and_type_checks(self):
        for html in (None, b"<p>Text</p>"):
            with self.subTest(html=html), self.assertRaises(PolicyDeltaError):
                PolicyDocument.from_html(html, role="old")
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_html("<p>too much text</p>", role="old",
                                     settings=Settings(max_document_chars=3))
        with self.assertRaises(PolicyDeltaError):
            PolicyDocument.from_html("<p>123456</p>", role="old",
                                     settings=Settings(max_document_chars=5))

    def test_pdf_ingestion_and_evidence(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "policy.PDF"
            with pymupdf.open() as pdf:
                page = pdf.new_page()
                page.insert_text((72, 72), "We retain records for 90 days.")
                pdf.save(str(path))
            document = PolicyDocument.from_pdf(path, role="new")
        self.assertIn("We retain records for 90 days.", document.text)
        start = document.text.index("90 days")
        verify_evidence(document.excerpt(start, start + len("90 days")), document)

    def test_pdf_page_order_and_empty_pages(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "policy.pdf"
            with pymupdf.open() as pdf:
                pdf.new_page().insert_text((72, 72), "Data collection")
                pdf.new_page()
                pdf.new_page().insert_text((72, 72), "Data deletion")
                pdf.save(str(path))
            document = PolicyDocument.from_pdf(path, role="old")
        self.assertEqual(document.text, "Data collection\n\nData deletion")

    def test_pdf_rejects_image_only_document(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "scan.pdf"
            with pymupdf.open() as pdf:
                pdf.new_page()
                pdf.save(str(path))
            with self.assertRaisesRegex(PolicyDeltaError, "extractable text"):
                PolicyDocument.from_pdf(path, role="old")

    def test_pdf_rejects_encrypted_document(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "protected.pdf"
            with pymupdf.open() as pdf:
                pdf.new_page().insert_text((72, 72), "Restricted")
                pdf.save(str(path), encryption=pymupdf.PDF_ENCRYPT_AES_256,
                         user_pw="secret", owner_pw="owner")
            with self.assertRaisesRegex(PolicyDeltaError, "Password-protected"):
                PolicyDocument.from_pdf(path, role="old")

    def test_pdf_invalid_paths_and_data(self):
        with tempfile.TemporaryDirectory() as directory:
            corrupt = Path(directory) / "corrupt.pdf"
            corrupt.write_bytes(b"not a PDF")
            for path in (None, Path(directory) / "missing.pdf",
                         Path(directory) / "policy.txt", corrupt):
                with self.subTest(path=path), self.assertRaises(PolicyDeltaError):
                    PolicyDocument.from_pdf(path, role="old")

    def test_pdf_resource_limits(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "policy.pdf"
            with pymupdf.open() as pdf:
                pdf.new_page().insert_text((72, 72), "Data retention is 180 days")
                pdf.new_page().insert_text((72, 72), "Data deletion")
                pdf.save(str(path))
            for settings in (
                Settings(max_pdf_pages=1),
                Settings(max_pdf_bytes=10),
                Settings(max_document_chars=5),
            ):
                with self.subTest(settings=settings), self.assertRaises(PolicyDeltaError):
                    PolicyDocument.from_pdf(path, role="new", settings=settings)

    def test_normalization_preserves_meaningful_terms(self):
        raw = "We do not share \n personal data for 180 days."
        self.assertEqual(normalize_policy_text(raw),
                         "We do not share personal data for 180 days.")

    def test_sentence_segmentation_and_exact_offsets(self):
        document = PolicyDocument.from_text(
            "We collect email addresses. We do not share them!",
            role="old")
        clauses = segment_policy(document)
        self.assertEqual(len(clauses), 2)
        self.assertEqual(clauses[1].text, "We do not share them!")
        for clause in clauses:
            self.assertEqual(document.text[clause.start:clause.end], clause.text)
            verify_evidence(document.excerpt(clause.start, clause.end), document)

    def test_headings_and_multiline_context(self):
        document = PolicyDocument.from_text(
            "Privacy Policy\n\nData Retention\nWe retain records\n"
            "for 30 days. We delete them on request.\n\n"
            "Data Sharing\nWe do not share data.", role="new")
        clauses = segment_policy(document)
        self.assertEqual([c.section for c in clauses],
                         ["Data Retention", "Data Retention", "Data Sharing"])
        self.assertEqual(clauses[0].normalized_text,
                         "We retain records for 30 days.")
        self.assertIn("\n", clauses[0].text)

    def test_abbreviations_and_quantities(self):
        document = PolicyDocument.from_text(
            "We use e.g. device IDs in the U.S. We retain them for 3.5 years. "
            "You can request deletion.", role="old")
        clauses = segment_policy(document)
        self.assertEqual(len(clauses), 3)
        self.assertIn("3.5 years", clauses[1].text)
        self.assertIn("e.g.", clauses[0].text)

    def test_bullet_items_and_paragraph_tracking(self):
        document = PolicyDocument.from_text(
            "Your Rights\n- Access your data\n- Delete your data\n\n"
            "We respond within 30 days.", role="new")
        clauses = segment_policy(document)
        self.assertEqual(len(clauses), 3)
        self.assertEqual([c.paragraph_index for c in clauses], [0, 1, 2])
        self.assertTrue(all(c.section == "Your Rights" for c in clauses))
        self.assertEqual(clauses[1].text, "- Delete your data")

    def test_html_integration(self):
        document = PolicyDocument.from_html(
            "<main><h2>Data Sharing</h2><p>We may share data. "
            "We require consent.</p></main>", role="new")
        clauses = segment_policy(document)
        self.assertEqual([c.section for c in clauses], ["Data Sharing"] * 2)
        self.assertEqual(clauses[1].text, "We require consent.")

    def test_sentence_case_headings(self):
        document = PolicyDocument.from_text(
            "Data retention\n\nWe retain information for 90 days.\n\n"
            "How we use your data\n\nWe analyze usage statistics.", role="new")
        clauses = segment_policy(document)
        self.assertEqual([clause.section for clause in clauses],
                         ["Data retention", "How we use your data"])

    def test_unpunctuated_privacy_statement_not_heading(self):
        document = PolicyDocument.from_text(
            "We collect device data\n\nWe share it with processors", role="old")
        self.assertEqual(len(segment_policy(document)), 2)

    def test_heading_only_document(self):
        document = PolicyDocument.from_text("Privacy Policy\n\nData Sharing", role="new")
        with self.assertRaisesRegex(PolicyDeltaError, "no analyzable clauses"):
            segment_policy(document)

    def test_unicode_and_crlf_offsets(self):
        document = PolicyDocument.from_text(
            "Données ✓ are stored.\r\nWe delete them after 30 days.", role="old")
        clauses = segment_policy(document)
        self.assertEqual(len(clauses), 2)
        self.assertEqual(clauses[0].start, 0)
        self.assertEqual(clauses[1].text, "We delete them after 30 days.")
        for clause in clauses:
            verify_evidence(document.excerpt(clause.start, clause.end), document)

    def test_identifiers_are_stable(self):
        document = PolicyDocument.from_text("One sentence. Another sentence.", role="new")
        self.assertEqual([c.clause_id for c in segment_policy(document)],
                         [c.clause_id for c in segment_policy(document)])

    def test_reproducible_seed(self):
        initialize(Settings(seed=7))
        first = random.random()
        initialize(Settings(seed=7))
        self.assertEqual(first, random.random())
        initialize()


    def test_alignment_reordered_clauses(self):
        old = PolicyDocument.from_text(
            "We retain data for 30 days. We share statistics with processors.", role="old")
        new = PolicyDocument.from_text(
            "We share statistics with processors. We retain data for 30 days.", role="new")
        alignment = align_policies(old, new)
        self.assertEqual([match.method for match in alignment.matches], ["exact", "exact"])
        self.assertEqual([match.new.text for match in alignment.matches],
                         ["We retain data for 30 days.",
                          "We share statistics with processors."])
        self.assertFalse(alignment.unmatched_old or alignment.unmatched_new)

    def test_alignment_rephrased_clause(self):
        old = PolicyDocument.from_text(
            "We collect email addresses for account verification.", role="old")
        new = PolicyDocument.from_text(
            "We collect email addresses for identity verification.", role="new")
        match, = align_policies(old, new).matches
        self.assertEqual(match.method, "tfidf")
        self.assertGreaterEqual(match.similarity, 0.4)
        verify_evidence(old.excerpt(match.old.start, match.old.end), old)
        verify_evidence(new.excerpt(match.new.start, match.new.end), new)

    def test_alignment_strong_paraphrase_not_forced(self):
        old = PolicyDocument.from_text(
            "We collect email addresses for account verification.", role="old")
        new = PolicyDocument.from_text(
            "We collect email addresses to verify accounts.", role="new")
        result = align_policies(old, new)
        self.assertFalse(result.matches)
        self.assertEqual(len(result.unmatched_old), 1)
        self.assertEqual(len(result.unmatched_new), 1)

    def test_alignment_additions_and_removals(self):
        old = PolicyDocument.from_text(
            "We retain records for 30 days. We never collect biometric data.", role="old")
        new = PolicyDocument.from_text(
            "We retain records for 30 days. We contact users by email.", role="new")
        result = align_policies(old, new)
        self.assertEqual(len(result.matches), 1)
        self.assertEqual(result.unmatched_old[0].text,
                         "We never collect biometric data.")
        self.assertEqual(result.unmatched_new[0].text,
                         "We contact users by email.")

    def test_alignment_duplicate_clauses_not_reused(self):
        old = PolicyDocument.from_text(
            "We delete old records. We delete old records.", role="old")
        new = PolicyDocument.from_text("We delete old records.", role="new")
        result = align_policies(old, new)
        self.assertEqual(len(result.matches), 1)
        self.assertEqual(len(result.unmatched_old), 1)
        self.assertFalse(result.unmatched_new)

    def test_alignment_threshold_can_prevent_match(self):
        old = PolicyDocument.from_text(
            "We collect email addresses to verify accounts.", role="old")
        new = PolicyDocument.from_text(
            "We collect email addresses for account verification.", role="new")
        result = align_policies(old, new, minimum_similarity=1.0)
        self.assertFalse(result.matches)
        self.assertEqual(len(result.unmatched_old), 1)
        self.assertEqual(len(result.unmatched_new), 1)

    def test_alignment_validates_arguments(self):
        old = PolicyDocument.from_text("We retain records.", role="old")
        new = PolicyDocument.from_text("We retain records.", role="new")
        for value in (0, -1, 1.1, True, float("nan"), float("inf")):
            with self.subTest(threshold=value), self.assertRaises(PolicyDeltaError):
                align_policies(old, new, minimum_similarity=value)
        for value in (0, -1, 1.5, True):
            with self.subTest(candidate_count=value), self.assertRaises(PolicyDeltaError):
                align_policies(old, new, candidates_per_clause=value)
        with self.assertRaises(PolicyDeltaError):
            align_policies(new, old)

    def test_alignment_case_whitespace_exact_match(self):
        old = PolicyDocument.from_text("We retain   records for 30 days.", role="old")
        new = PolicyDocument.from_text("we retain records for 30 days.", role="new")
        match, = align_policies(old, new).matches
        self.assertEqual(match.method, "exact")
        self.assertEqual(match.similarity, 1.0)

    def test_alignment_is_deterministic(self):
        old = PolicyDocument.from_text(
            "We share device data. We retain records for 30 days.", role="old")
        new = PolicyDocument.from_text(
            "We retain records for 90 days. We share device statistics.", role="new")
        first, second = align_policies(old, new), align_policies(old, new)
        self.assertEqual(first, second)

    def test_tfidf_similarity_uses_lexical_evidence(self):
        old = PolicyDocument.from_text("We do not share personal data.", role="old")
        new = PolicyDocument.from_text("We may share personal data.", role="new")
        scores = tfidf_similarity(segment_policy(old), segment_policy(new))
        self.assertEqual(scores.shape, (1, 1))
        self.assertGreater(scores[0, 0], 0)
        self.assertLess(scores[0, 0], 1)

    def test_tfidf_similarity_empty_vocab_and_inputs(self):
        self.assertEqual(tfidf_similarity([], []).shape, (0, 0))
        self.assertEqual(tfidf_similarity([], segment_policy(self.document)).shape,
                         (0, 1))
        old = [PolicyClause("x", "old", "...", 0, 3, None, 0)]
        new = [PolicyClause("y", "new", "!!!", 0, 3, None, 0)]
        self.assertEqual(tfidf_similarity(old, new)[0, 0], 0)

    def test_alignment_across_input_formats(self):
        old = PolicyDocument.from_html(
            "<main><h2>Data Sharing</h2><p>We disclose data to vendors.</p></main>",
            role="old")
        new = PolicyDocument.from_text(
            "Data Sharing\n\nWe disclose data to vendors.", role="new")
        result = align_policies(old, new)
        self.assertEqual(len(result.matches), 1)
        self.assertEqual(result.matches[0].method, "exact")


    def test_minilm_alignment_matches_paraphrase(self):
        class Encoder:
            def encode(self, texts, **kwargs):
                return np.array([[1.0, 0.0] if "information" in t
                                 else [2.0, 0.0] for t in texts])

        old = PolicyDocument.from_text(
            "We collect information to authenticate your account.", role="old")
        new = PolicyDocument.from_text(
            "Your details are gathered for identity confirmation.", role="new")
        self.assertFalse(align_policies(old, new).matches)
        match, = align_policies(old, new, method="minilm", encoder=Encoder()).matches
        self.assertEqual(match.method, "minilm")
        self.assertAlmostEqual(match.similarity, 1.0)
        verify_evidence(old.excerpt(match.old.start, match.old.end), old)
        verify_evidence(new.excerpt(match.new.start, match.new.end), new)

    def test_minilm_preserves_exact_priority(self):
        old = PolicyDocument.from_text("We delete records after 30 days.", role="old")
        new = PolicyDocument.from_text("We delete records after 30 days.", role="new")
        match, = align_policies(old, new, method="minilm").matches
        self.assertEqual(match.method, "exact")

    def test_minilm_empty_input_does_not_load_model(self):
        clauses = segment_policy(self.document)
        self.assertEqual(minilm_similarity([], clauses).shape, (0, 1))
        self.assertEqual(minilm_similarity(clauses, []).shape, (1, 0))

    def test_minilm_rejects_invalid_vectors(self):
        class Encoder:
            def __init__(self, output):
                self.output = output

            def encode(self, texts, **kwargs):
                return self.output

        clauses = segment_policy(self.document)
        for output in (np.array([[1.0], [float("nan")]]),
                       np.array([[0.0], [0.0]]), np.array([1.0, 2.0])):
            with self.subTest(output=output.shape), self.assertRaises(PolicyDeltaError):
                minilm_similarity(clauses, clauses, encoder=Encoder(output))

    def test_minilm_rejects_invalid_method(self):
        with self.assertRaises(PolicyDeltaError):
            align_policies(self.document, PolicyDocument.from_text(
                self.document.text, role="new"), method="unknown")

    def test_minilm_preserves_one_to_one_alignment(self):
        class Encoder:
            def encode(self, texts, **kwargs):
                return np.array([[1., 0.], [1., 0.], [1., 0.]])

        old = PolicyDocument.from_text(
            "We erase customer profiles. We retain limited transaction records.", role="old")
        new = PolicyDocument.from_text(
            "User accounts are removed from storage.", role="new")
        result = align_policies(old, new, method="minilm", encoder=Encoder())
        self.assertEqual(len(result.matches), 1)
        self.assertEqual(len(result.unmatched_old), 1)
        self.assertEqual(len(result.unmatched_new), 0)


    @staticmethod
    def _paired_changes(old_text, new_text, **kwargs):
        old = PolicyDocument.from_text(old_text, role="old")
        new = PolicyDocument.from_text(new_text, role="new")
        alignment = ClauseAlignment((ClauseMatch(segment_policy(old)[0],
                                                 segment_policy(new)[0], 0.9, "minilm"),), (), ())
        return analyze_changes(alignment, **kwargs)

    def test_changes_unchanged_without_model(self):
        old = PolicyDocument.from_text("We retain records.", role="old")
        new = PolicyDocument.from_text("We retain records.", role="new")
        result = analyze_changes(align_policies(old, new))
        self.assertEqual([entry.operation for entry in result], ["UNCHANGED"])
        self.assertIsNone(result[0].old_to_new)

    def test_changes_unmatched_clauses(self):
        old = PolicyDocument.from_text("We retain records.", role="old")
        new = PolicyDocument.from_text("We collect device identifiers.", role="new")
        result = analyze_changes(ClauseAlignment((), tuple(segment_policy(old)),
                                                  tuple(segment_policy(new))))
        self.assertEqual([entry.operation for entry in result], ["REMOVED", "ADDED"])
        verify_evidence(old.excerpt(result[0].old.start, result[0].old.end), old)
        verify_evidence(new.excerpt(result[1].new.start, result[1].new.end), new)

    def test_changes_detect_numeric_shift(self):
        self.assertEqual(change_signals("We retain data for 30 days.",
                                        "We retain data for 90 days."), ("quantity",))
        self.assertEqual(change_signals("We retain data for thirty days.",
                                        "We retain data for 30 days."), ())
        self.assertEqual(change_signals("We retain data for 30 days.",
                                        "We retain data for 30 months."), ("quantity",))

    def test_changes_detect_negation_and_modality(self):
        self.assertEqual(change_signals("We do not share information.",
                                        "We may share information."),
                         ("negation", "modality"))
        self.assertEqual(change_signals("We will store data.", "We may store data."),
                         ("modality",))

    def test_changes_equivalent_paraphrase(self):
        class Model:
            def predict(self, pairs, **kwargs):
                return np.array([[0, 7, 0], [0, 7, 0]], dtype=float)

        result, = self._paired_changes("We delete account records after thirty days.",
                                       "Account records are deleted after 30 days.",
                                       nli_model=Model())
        self.assertEqual(result.operation, "REPHRASED")
        self.assertEqual(result.signals, ())
        self.assertGreater(result.old_to_new[1], 0.9)

    def test_changes_bidirectional_asymmetry(self):
        class Model:
            def predict(self, pairs, **kwargs):
                return np.array([[0, 7, 0], [0, 0, 7]])

        result, = self._paired_changes("We collect device identifiers.",
                                       "We collect device identifiers for advertising.",
                                       nli_model=Model())
        self.assertEqual(result.operation, "MODIFIED")
        self.assertGreater(result.old_to_new[1], 0.9)
        self.assertGreater(result.new_to_old[2], 0.9)

    def test_changes_contradiction(self):
        class Model:
            def predict(self, pairs, **kwargs):
                return [[7, 0, 0], [7, 0, 0]]

        result, = self._paired_changes("We sell personal information.",
                                       "We never sell personal information.",
                                       nli_model=Model())
        self.assertEqual(result.operation, "MODIFIED")
        self.assertIn("negation", result.signals)

    def test_changes_uncertain_neutral(self):
        class Model:
            def predict(self, pairs, **kwargs):
                return [[0, 0, 7], [0, 0, 7]]

        result, = self._paired_changes("We protect information with encryption.",
                                       "We store information on secure servers.",
                                       nli_model=Model())
        self.assertEqual(result.operation, "UNCERTAIN")

    def test_changes_rule_signal_even_if_nli_entails(self):
        class Model:
            def predict(self, pairs, **kwargs):
                return [[0, 7, 0], [0, 7, 0]]

        result, = self._paired_changes("We retain records for 30 days.",
                                       "We retain records for 90 days.",
                                       nli_model=Model())
        self.assertEqual(result.operation, "MODIFIED")
        self.assertIn("quantity", result.signals)

    def test_changes_rejects_invalid_model_outputs(self):
        class Model:
            def __init__(self, value):
                self.value = value

            def predict(self, pairs, **kwargs):
                return self.value

        for output in (np.zeros((1, 3)), np.zeros((2, 2)),
                       np.full((2, 3), float("nan"))):
            with self.subTest(output=output.shape), self.assertRaises(PolicyDeltaError):
                self._paired_changes("We collect information.",
                                     "We store information.", nli_model=Model(output))

    def test_changes_validates_threshold(self):
        for value in (0.0, 0.5, 1.0, True, float("nan")):
            with self.subTest(value=value), self.assertRaises(PolicyDeltaError):
                analyze_changes(ClauseAlignment((), (), ()), confidence_threshold=value)

    def test_changes_long_clause_requires_review(self):
        class Model:
            def predict(self, pairs, **kwargs):
                raise AssertionError("Long input must not be inferred")

        old = "We collect " + "location " * 165 + "data."
        new = "We store " + "location " * 165 + "data."
        result, = self._paired_changes(old, new, nli_model=Model())
        self.assertEqual(result.operation, "UNCERTAIN")
        self.assertIn("input_too_long", result.signals)

    def test_changes_handles_tokenizer_limits(self):
        class Model:
            max_seq_length = 32

            @staticmethod
            def tokenizer(old, new, **kwargs):
                return {"input_ids": list(range(40))}

            def predict(self, pairs, **kwargs):
                raise AssertionError("Truncated input should not be inferred")

        result, = self._paired_changes("We store contact data.",
                                       "We collect contact data.", nli_model=Model())
        self.assertEqual(result.operation, "UNCERTAIN")
        self.assertIn("input_too_long", result.signals)

    def test_changes_surfaces_inference_errors(self):
        class Model:
            def predict(self, pairs, **kwargs):
                raise RuntimeError("Model unavailable")

        with self.assertRaisesRegex(PolicyDeltaError, "NLI inference failed"):
            self._paired_changes("We store contact data.", "We collect contact data.",
                                 nli_model=Model())

    def test_changes_integration_with_alignment(self):
        old = PolicyDocument.from_text("We retain records for 30 days.", role="old")
        new = PolicyDocument.from_text("We retain records for 90 days.", role="new")

        class Model:
            def predict(self, pairs, **kwargs):
                return [[0, 0, 7], [0, 0, 7]]

        findings = analyze_changes(align_policies(old, new), nli_model=Model())
        self.assertEqual(len(findings), 1)
        self.assertEqual(findings[0].operation, "MODIFIED")
        verify_evidence(old.excerpt(findings[0].old.start, findings[0].old.end), old)


    def test_privacy_topics_single_categories(self):
        examples = {
            "data_collection": "We collect account information.",
            "data_use": "We use personal information for advertising.",
            "third_party_sharing": "We disclose personal data to advertising partners.",
            "data_retention": "We retain account records for 90 days.",
            "user_rights": "You can request deletion of your information.",
            "data_security": "We encrypt sensitive records.",
            "cookies_tracking": "We use cookies for analytics.",
            "international_transfers": "We transfer data outside the EU.",
            "sensitive_data": "We collect biometric data.",
            "children": "Our service is not for children under 13.",
            "policy_updates": "We may update this privacy policy.",
        }
        for category, passage in examples.items():
            with self.subTest(category=category):
                self.assertIn(category, classify_privacy_topics(passage))

    def test_privacy_topics_multilabel(self):
        topics = classify_privacy_topics(
            "We collect device identifiers and share personal data with advertisers."
        )
        self.assertIn("data_collection", topics)
        self.assertIn("third_party_sharing", topics)
        self.assertEqual(len(topics), len(set(topics)))

    def test_privacy_topics_combine_old_and_new(self):
        categories = classify_privacy_topics(
            "We retain account records for 30 days.",
            "We share your information with third parties.",
        )
        self.assertIn("data_retention", categories)
        self.assertIn("third_party_sharing", categories)

    def test_privacy_topics_passive_and_reversed_phrasing(self):
        cases = (
            ("Personal information we collect includes location data.", "data_collection"),
            ("We receive information about device activity.", "data_collection"),
            ("We protect your data with technical safeguards.", "data_security"),
            ("Your account information is retained for two years.", "data_retention"),
        )
        for passage, category in cases:
            with self.subTest(passage=passage):
                self.assertIn(category, classify_privacy_topics(passage))

    def test_privacy_topics_no_cross_passage_matches(self):
        self.assertEqual(classify_privacy_topics("We collect.", "Information is public."),
                         ("other",))

    def test_privacy_topics_unknown_and_invalid(self):
        self.assertEqual(classify_privacy_topics("Contact us for questions."), ("other",))
        with self.assertRaises(PolicyDeltaError):
            classify_privacy_topics()
        with self.assertRaises(PolicyDeltaError):
            classify_privacy_topics(None)

    def test_privacy_topics_are_case_insensitive(self):
        self.assertIn("third_party_sharing",
                      classify_privacy_topics("WE SHARE PERSONAL DATA WITH ADVERTISERS."))

    def test_privacy_topics_integrated_on_matched_changes(self):
        old = PolicyDocument.from_text("We retain records for 30 days.", role="old")
        new = PolicyDocument.from_text("We retain records for 90 days.", role="new")

        class Model:
            def predict(self, pairs, **kwargs):
                return [[0, 0, 7], [0, 0, 7]]

        finding, = analyze_changes(align_policies(old, new), nli_model=Model())
        self.assertEqual(finding.operation, "MODIFIED")
        self.assertIn("data_retention", finding.categories)
        self.assertEqual(old.text[finding.old.start:finding.old.end], finding.old.text)
        self.assertEqual(new.text[finding.new.start:finding.new.end], finding.new.text)

    def test_privacy_topics_integrated_on_unmatched(self):
        old = PolicyDocument.from_text("We store account records for one year.", role="old")
        new = PolicyDocument.from_text("We use cookies to track visitors.", role="new")
        findings = analyze_changes(ClauseAlignment((), tuple(segment_policy(old)),
                                                  tuple(segment_policy(new))))
        self.assertEqual([finding.operation for finding in findings], ["REMOVED", "ADDED"])
        self.assertIn("data_retention", findings[0].categories)
        self.assertIn("cookies_tracking", findings[1].categories)

    def test_privacy_topics_unchanged_without_nli(self):
        old = PolicyDocument.from_text("We use cookies to track visitors.", role="old")
        new = PolicyDocument.from_text("We use cookies to track visitors.", role="new")
        finding, = analyze_changes(align_policies(old, new))
        self.assertEqual(finding.operation, "UNCHANGED")
        self.assertIn("cookies_tracking", finding.categories)

    def test_privacy_topics_priority_is_stable(self):
        passage = "We share personal data with advertisers and collect device information."
        first = classify_privacy_topics(passage)
        self.assertEqual(first, classify_privacy_topics(passage))
        self.assertEqual(first, ("data_collection", "third_party_sharing"))



    def test_assessment_restriction_replaced_by_permission(self):
        old = PolicyDocument.from_text(
            "We do not share personal data with advertisers.", role="old"
        )
        new = PolicyDocument.from_text(
            "We may share personal data with advertisers.", role="new"
        )
        findings = (ChangeRecord("MODIFIED", segment_policy(old)[0],
                                 segment_policy(new)[0], ("negation", "modality")),)
        result, = assess_changes(findings, old, new)
        self.assertEqual(result.significance, "high")
        self.assertIn("sharing_restriction_relaxed", result.factors)
        self.assertEqual([e.text for e in result.evidence], [old.text, new.text])
        for span, document in zip(result.evidence, (old, new)):
            verify_evidence(span, document)

    def test_assessment_retention_period_change(self):
        old = PolicyDocument.from_text("We retain records for 30 days.", role="old")
        new = PolicyDocument.from_text("We retain records for 90 days.", role="new")
        finding = ChangeRecord("MODIFIED", segment_policy(old)[0], segment_policy(new)[0])
        result, = assess_changes((finding,), old, new)
        self.assertEqual(result.significance, "medium")
        self.assertIn("quantity", result.factors)
        self.assertIn("quantity", result.explanation)

    def test_assessment_unchanged_and_rephrased(self):
        old = PolicyDocument.from_text("We use cookies for analytics.", role="old")
        new = PolicyDocument.from_text("We use cookies for analytics.", role="new")
        finding = ChangeRecord("UNCHANGED", segment_policy(old)[0], segment_policy(new)[0])
        result, = assess_changes((finding,), old, new)
        self.assertEqual(result.significance, "none")
        self.assertEqual(result.factors, ())
        finding = ChangeRecord("REPHRASED", segment_policy(old)[0], segment_policy(new)[0])
        result, = assess_changes((finding,), old, new)
        self.assertEqual(result.significance, "low")

    def test_assessment_uncertain_is_not_given_significance(self):
        old = PolicyDocument.from_text("We collect contact information.", role="old")
        new = PolicyDocument.from_text("We use contact information.", role="new")
        finding = ChangeRecord("UNCERTAIN", segment_policy(old)[0],
                               segment_policy(new)[0], ("input_too_long",))
        result, = assess_changes((finding,), old, new)
        self.assertEqual(result.significance, "uncertain")
        self.assertIn("supported inference length", result.explanation)

    def test_assessment_added_removed_and_no_unjustified_high(self):
        old = PolicyDocument.from_text("We retain account records for 30 days.", role="old")
        new = PolicyDocument.from_text("We collect location information.", role="new")
        findings = (
            ChangeRecord("REMOVED", segment_policy(old)[0], None),
            ChangeRecord("ADDED", None, segment_policy(new)[0]),
        )
        results = assess_changes(findings, old, new)
        self.assertEqual([a.significance for a in results], ["medium", "medium"])
        self.assertEqual([len(a.evidence) for a in results], [1, 1])
        self.assertIn("no match", results[0].explanation)
        self.assertIn("no match", results[1].explanation)

    def test_assessment_unmatched_restriction_is_review_priority(self):
        old = PolicyDocument.from_text(
            "We do not share personal data with advertisers.", role="old"
        )
        new = PolicyDocument.from_text("We use cookies for analytics.", role="new")
        finding = ChangeRecord("REMOVED", segment_policy(old)[0], None)
        result, = assess_changes((finding,), old, new)
        self.assertEqual(result.significance, "high")
        self.assertIn("moved or rephrased", result.explanation)

    def test_assessment_does_not_overstate_nonsharing_negation(self):
        old = PolicyDocument.from_text("We do not retain account data.", role="old")
        new = PolicyDocument.from_text("We may retain account data.", role="new")
        finding = ChangeRecord("MODIFIED", segment_policy(old)[0], segment_policy(new)[0])
        result, = assess_changes((finding,), old, new)
        self.assertEqual(result.significance, "medium")

    def test_assessment_rejects_wrong_document_and_tampered_span(self):
        old = PolicyDocument.from_text("We collect account information.", role="old")
        new = PolicyDocument.from_text("We use account information.", role="new")
        clauses = (segment_policy(old)[0], segment_policy(new)[0])
        finding = ChangeRecord("MODIFIED", *clauses)
        different = PolicyDocument.from_text("Other content.", role="old")
        with self.assertRaises(EvidenceError):
            assess_changes((finding,), different, new)
        with self.assertRaises(EvidenceError):
            assess_changes((replace(finding, old=replace(clauses[0], text="Altered text.")),),
                           old, new)
        with self.assertRaises(EvidenceError):
            assess_changes((replace(finding, old=replace(clauses[0], start=-1)),), old, new)

    def test_assessment_rejects_inconsistent_operations(self):
        old = PolicyDocument.from_text("We collect account information.", role="old")
        new = PolicyDocument.from_text("We use account information.", role="new")
        old_clause, new_clause = segment_policy(old)[0], segment_policy(new)[0]
        for finding in (
            ChangeRecord("ADDED", old_clause, new_clause),
            ChangeRecord("REMOVED", None, new_clause),
            ChangeRecord("MODIFIED", old_clause, None),
        ):
            with self.subTest(operation=finding.operation), self.assertRaises(PolicyDeltaError):
                assess_changes((finding,), old, new)

    def test_assessment_empty_input_and_reproducible_output(self):
        old = PolicyDocument.from_text("We collect account information.", role="old")
        new = PolicyDocument.from_text("We use account information.", role="new")
        finding = ChangeRecord("MODIFIED", segment_policy(old)[0], segment_policy(new)[0])
        self.assertEqual(assess_changes((), old, new), ())
        self.assertEqual(assess_changes((finding,), old, new),
                         assess_changes((finding,), old, new))

    def test_assessment_with_analyze_changes(self):
        old = PolicyDocument.from_text("We retain records for 30 days.", role="old")
        new = PolicyDocument.from_text("We retain records for 90 days.", role="new")

        class Model:
            def predict(self, pairs, **kwargs):
                return [[0, 0, 7], [0, 0, 7]]

        findings = analyze_changes(align_policies(old, new), nli_model=Model())
        result, = assess_changes(findings, old, new)
        self.assertEqual(result.significance, "medium")
        self.assertIn("data_retention", result.finding.categories)
        self.assertEqual(len(result.evidence), 2)


    def test_report_summary_and_verified_evidence(self):
        old = PolicyDocument.from_text("We retain records for 30 days.", role="old")
        new = PolicyDocument.from_text("We retain records for 90 days.", role="new")
        finding = ChangeRecord("MODIFIED", segment_policy(old)[0], segment_policy(new)[0],
                               ("quantity",), categories=("data_retention",))
        assessments = assess_changes((finding,), old, new)
        report = build_comparison_report(assessments, old, new)
        self.assertEqual(report["summary"]["total_findings"], 1)
        self.assertEqual(report["summary"]["review_findings"], 1)
        self.assertEqual(report["summary"]["by_operation"]["MODIFIED"], 1)
        self.assertEqual(report["summary"]["by_significance"]["medium"], 1)
        self.assertEqual(report["summary"]["by_topic"]["data_retention"], 1)
        self.assertEqual(report["findings"][0]["evidence"]["old"]["text"], old.text)
        self.assertEqual(report["documents"]["new"]["sha256"], new.sha256)

    def test_report_includes_added_and_removed(self):
        old = PolicyDocument.from_text("We retain records for 30 days.", role="old")
        new = PolicyDocument.from_text("We collect device information.", role="new")
        findings = (ChangeRecord("REMOVED", segment_policy(old)[0], None),
                    ChangeRecord("ADDED", None, segment_policy(new)[0]))
        report = build_comparison_report(assess_changes(findings, old, new), old, new)
        self.assertEqual(report["summary"]["by_operation"]["ADDED"], 1)
        self.assertEqual(report["summary"]["by_operation"]["REMOVED"], 1)
        self.assertIsNone(report["findings"][0]["evidence"]["new"])
        self.assertIsNone(report["findings"][1]["evidence"]["old"])
        self.assertEqual(len(list(csv.DictReader(io.StringIO(report_csv(report))))), 2)

    def test_report_empty_results_have_stable_counts(self):
        old = PolicyDocument.from_text("We collect usage data.", role="old")
        new = PolicyDocument.from_text("We collect usage data.", role="new")
        report = build_comparison_report((), old, new)
        self.assertEqual(report["summary"]["total_findings"], 0)
        self.assertEqual(report["summary"]["by_topic"], {})
        self.assertEqual(sum(report["summary"]["by_operation"].values()), 0)
        self.assertEqual(len(list(csv.DictReader(io.StringIO(report_csv(report))))), 0)

    def test_report_json_keeps_unicode_and_is_deterministic(self):
        old = PolicyDocument.from_text("We collect café information.", role="old")
        new = PolicyDocument.from_text("We collect café information.", role="new")
        report = build_comparison_report(
            assess_changes(analyze_changes(align_policies(old, new)), old, new), old, new
        )
        result = report_json(report)
        self.assertEqual(json.loads(result), report)
        self.assertIn("café", result)
        self.assertEqual(result, report_json(report))
        self.assertEqual(report["summary"]["by_significance"]["none"], 1)

    def test_report_csv_escapes_quoted_and_multiline_text(self):
        text = 'We collect "contact, phone" information.\nWe also collect device data.'
        old = PolicyDocument.from_text(text, role="old")
        new = PolicyDocument.from_text(text, role="new")
        report = build_comparison_report(
            assess_changes(analyze_changes(align_policies(old, new)), old, new), old, new
        )
        records = list(csv.DictReader(io.StringIO(report_csv(report))))
        self.assertEqual(len(records), 2)
        self.assertEqual(records[0]["old_text"], 'We collect "contact, phone" information.')
        self.assertEqual(records[1]["new_text"], "We also collect device data.")

    def test_report_csv_prevents_formula_execution(self):
        old = PolicyDocument.from_text('=HYPERLINK("https://example.com")', role="old")
        new = PolicyDocument.from_text(old.text, role="new")
        report = build_comparison_report(
            assess_changes(analyze_changes(align_policies(old, new)), old, new), old, new
        )
        first = next(csv.DictReader(io.StringIO(report_csv(report))))
        self.assertTrue(first["old_text"].startswith("'="))
        self.assertEqual(report["findings"][0]["evidence"]["old"]["text"], old.text)

    def test_report_rejects_modified_assessment_and_wrong_source(self):
        old = PolicyDocument.from_text("We share personal data with advertisers.", role="old")
        new = PolicyDocument.from_text("We may share personal data with advertisers.", role="new")
        finding = ChangeRecord("MODIFIED", segment_policy(old)[0], segment_policy(new)[0])
        assessment, = assess_changes((finding,), old, new)
        with self.assertRaises(EvidenceError):
            build_comparison_report((replace(assessment, explanation="Unverified"),), old, new)
        other = PolicyDocument.from_text("Other privacy text.", role="old")
        with self.assertRaises(EvidenceError):
            build_comparison_report((assessment,), other, new)
        with self.assertRaises(PolicyDeltaError):
            build_comparison_report([assessment], old, new)

    def test_report_rejects_topic_mismatch(self):
        old = PolicyDocument.from_text("We use cookies for analytics.", role="old")
        new = PolicyDocument.from_text("We use cookies for analytics.", role="new")
        finding = ChangeRecord("UNCHANGED", segment_policy(old)[0], segment_policy(new)[0],
                               categories=("third_party_sharing",))
        with self.assertRaises(EvidenceError):
            build_comparison_report(assess_changes((finding,), old, new), old, new)

    def test_report_saved_files_match_serialized_content(self):
        old = PolicyDocument.from_text("We use cookies for analytics.", role="old")
        new = PolicyDocument.from_text(old.text, role="new")
        report = build_comparison_report(
            assess_changes(analyze_changes(align_policies(old, new)), old, new), old, new
        )
        with tempfile.TemporaryDirectory() as folder:
            json_path, csv_path = save_comparison_report(report, Path(folder) / "reports")
            self.assertEqual(json_path.read_text(encoding="utf-8"), report_json(report))
            self.assertEqual(csv_path.read_text(encoding="utf-8"), report_csv(report))
            self.assertEqual(json.loads(json_path.read_text(encoding="utf-8")), report)


    def test_ui_policy_sources(self):
        doc = read_policy_input("We collect information.", None, "old")
        self.assertEqual(doc.text, "We collect information.")
        with tempfile.TemporaryDirectory() as folder:
            source = Path(folder) / "policy.html"
            source.write_text("<main><p>We delete account data.</p></main>", encoding="utf-8")
            loaded = read_policy_input(None, str(source), "new")
            self.assertEqual(loaded.text, "We delete account data.")
            with self.assertRaises(PolicyDeltaError):
                read_policy_input("We collect information.", str(source), "new")
            with self.assertRaises(PolicyDeltaError):
                read_policy_input(None, str(Path(folder) / "bad.csv"), "new")

    def test_ui_identical_policy_comparison(self):
        report = compare_policy_inputs("We retain records for 30 days.", None,
                                       "We retain records for 30 days.", None,
                                       method="tfidf")
        self.assertEqual(report["summary"]["by_operation"]["UNCHANGED"], 1)
        self.assertEqual(report["summary"]["review_findings"], 0)

    def test_ui_changed_policy_with_mock_nli(self):
        class Model:
            def predict(self, pairs, **kwargs):
                return [[7, 0, 0], [7, 0, 0]]
        report = compare_policy_inputs(
            "We do not share data with advertisers.", None,
            "We may share data with advertisers.", None,
            method="tfidf", nli_model=Model(), minimum_similarity=0.2)
        self.assertEqual(report["summary"]["by_operation"]["MODIFIED"], 1)
        self.assertEqual(report["summary"]["by_significance"]["high"], 1)

    def test_ui_filtering_does_not_mutate_report(self):
        report = {"findings": [
            {"finding_id": "a", "significance": "high", "categories": ["data_retention"],
             "operation": "MODIFIED", "explanation": "Review.",
             "evidence": {"old": None, "new": None}},
            {"finding_id": "b", "significance": "none", "categories": ["other"],
             "operation": "UNCHANGED", "explanation": "Same.",
             "evidence": {"old": None, "new": None}},
        ]}
        snapshot = json.dumps(report, sort_keys=True)
        self.assertEqual([item["finding_id"] for item in filter_report(report, "high")], ["a"])
        self.assertEqual([item["finding_id"] for item in filter_report(report, topic="other")], ["b"])
        self.assertEqual([item["finding_id"] for item in filter_report(report, operation="MODIFIED")], ["a"])
        self.assertEqual(json.dumps(report, sort_keys=True), snapshot)

    def test_ui_escapes_evidence(self):
        report = {"findings": [{
            "finding_id": "a", "operation": "MODIFIED", "significance": "medium",
            "categories": ["other"], "explanation": "<script>alert(1)</script>",
            "evidence": {"old": {"section": None, "start": 0, "end": 10,
                                 "text": "<img src=x onerror=alert(1)>"}, "new": None},
        }]}
        rendered = finding_detail(report, "a")
        self.assertNotIn("<script>", rendered)
        self.assertNotIn("<img", rendered)
        self.assertIn("&lt;img", rendered)
        self.assertIn("&lt;script&gt;", rendered)

    def test_ui_empty_and_invalid_input_clears_outputs(self):
        results = run_comparison_ui("", None, "", None, "tfidf", 0.4, 0.7)
        self.assertIsNone(results[0])
        self.assertIn("Unable to compare", results[1])
        self.assertEqual(results[-2:], (None, None))
        self.assertEqual(filter_report(None), [])
        self.assertIn("Select a finding", finding_detail(None, None))

    def test_ui_download_paths(self):
        outputs = run_comparison_ui("We use cookies.", None, "We use cookies.", None,
                                    "tfidf", 0.4, 0.7)
        self.assertEqual(outputs[0]["summary"]["by_operation"]["UNCHANGED"], 1)
        self.assertEqual(json.loads(Path(outputs[-2]).read_text(encoding="utf-8")), outputs[0])
        self.assertTrue(Path(outputs[-1]).is_file())

    def test_ui_gradio_builds(self):
        app = build_policy_delta_app()
        self.assertIsInstance(app, gr.Blocks)



    def _sample_benchmark(self, directory: Path) -> Path:
        examples = [
            ("service-a", "calibration",
             "We retain records for 30 days.\n\nWe use cookies.",
             "We retain records for 90 days.\n\nWe use cookies."),
            ("service-b", "test",
             "We do not share data with advertisers.\n\nWe collect device data.",
             "We may share data with advertisers.\n\nWe collect device data."),
        ]
        manifest = directory / "benchmark.csv"
        with manifest.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=[
                "site", "pair_id", "split", "old_path", "new_path", "gold_matches", "source",
            ])
            writer.writeheader()
            for site, split, old_text, new_text in examples:
                (directory / f"{site}-old.txt").write_text(old_text, encoding="utf-8")
                (directory / f"{site}-new.txt").write_text(new_text, encoding="utf-8")
                writer.writerow({"site": site, "pair_id": f"{site}-001", "split": split,
                                 "old_path": f"{site}-old.txt", "new_path": f"{site}-new.txt",
                                 "gold_matches": json.dumps([[0, 0], [1, 1]]),
                                 "source": "synthetic test fixture"})
        return manifest

    def test_benchmark_loader_and_splits(self):
        with tempfile.TemporaryDirectory() as root:
            pairs = load_alignment_benchmark(self._sample_benchmark(Path(root)))
            self.assertEqual(len(pairs), 2)
            self.assertEqual({p.split for p in pairs}, {"calibration", "test"})
            self.assertEqual(pairs[0].gold_matches, frozenset({(0, 0), (1, 1)}))

    def test_calibration_and_heldout_metrics(self):
        with tempfile.TemporaryDirectory() as root:
            pairs = load_alignment_benchmark(self._sample_benchmark(Path(root)))
            fitted = calibrate_alignment_threshold(pairs, method="tfidf")
            heldout = evaluate_alignment_threshold(pairs, fitted)
            self.assertGreater(fitted["threshold"], 0)
            self.assertEqual(fitted["calibration_pairs"], 1)
            self.assertEqual(heldout["test_pairs"], 1)
            self.assertEqual(heldout["test_metrics"]["tp"] +
                             heldout["test_metrics"]["fn"], 2)
            self.assertNotIn("test_metrics", fitted)

    def test_calibration_changes_default_only_when_activated(self):
        with tempfile.TemporaryDirectory() as root:
            data = load_alignment_benchmark(self._sample_benchmark(Path(root)))
            calibration = calibrate_alignment_threshold(data)
        previous = CALIBRATED_MATCH_THRESHOLDS.copy()
        try:
            use_alignment_calibration(calibration)
            self.assertAlmostEqual(alignment_threshold("tfidf"), calibration["threshold"])
            self.assertEqual(alignment_threshold("minilm"),
                             previous.get("minilm", _PROVISIONAL_MATCH_THRESHOLDS["minilm"]))
        finally:
            CALIBRATED_MATCH_THRESHOLDS.clear()
            CALIBRATED_MATCH_THRESHOLDS.update(previous)

    def test_benchmark_rejects_site_leakage(self):
        with tempfile.TemporaryDirectory() as root:
            manifest = self._sample_benchmark(Path(root))
            text = manifest.read_text(encoding="utf-8").replace("service-b,service-b-001,test", "service-a,service-b-001,test")
            manifest.write_text(text, encoding="utf-8")
            with self.assertRaises(PolicyDeltaError):
                load_alignment_benchmark(manifest)

    def test_benchmark_rejects_document_leakage(self):
        with tempfile.TemporaryDirectory() as root:
            manifest = self._sample_benchmark(Path(root))
            (Path(root) / "service-b-old.txt").write_text(
                (Path(root) / "service-a-old.txt").read_text(encoding="utf-8"), encoding="utf-8")
            with self.assertRaises(PolicyDeltaError):
                load_alignment_benchmark(manifest)

    def test_benchmark_rejects_invalid_gold_and_path_escape(self):
        with tempfile.TemporaryDirectory() as root:
            manifest = self._sample_benchmark(Path(root))
            contents = manifest.read_text(encoding="utf-8")
            manifest.write_text(contents.replace('[[0, 0], [1, 1]]', '[[0, 0], [0, 1]]'),
                                encoding="utf-8")
            with self.assertRaises(PolicyDeltaError):
                load_alignment_benchmark(manifest)
            manifest.write_text(contents.replace("service-a-old.txt", "../secret.txt"), encoding="utf-8")
            with self.assertRaises(PolicyDeltaError):
                load_alignment_benchmark(manifest)

    def test_benchmark_rejects_heldout_calibration(self):
        with tempfile.TemporaryDirectory() as root:
            pairs = load_alignment_benchmark(self._sample_benchmark(Path(root)))
            with self.assertRaises(PolicyDeltaError):
                calibrate_alignment_threshold([p for p in pairs if p.split == "test"])
            with self.assertRaises(PolicyDeltaError):
                evaluate_alignment_threshold([p for p in pairs if p.split == "calibration"],
                                             {"method": "tfidf", "threshold": 0.5})

    def test_minilm_calibration_uses_encoder(self):
        class Encoder:
            def encode(self, texts, **kwargs):
                return np.asarray([[1.0, 0.0] if "retain" in text else [0.0, 1.0]
                                   for text in texts])

        with tempfile.TemporaryDirectory() as root:
            pairs = load_alignment_benchmark(self._sample_benchmark(Path(root)))
            fitted = calibrate_alignment_threshold(pairs, method="minilm", encoder=Encoder())
            results = evaluate_alignment_threshold(pairs, fitted, encoder=Encoder())
        self.assertEqual(fitted["method"], "minilm")
        self.assertEqual(results["test_metrics"]["tp"], 2)

    def test_ui_reports_provisional_threshold(self):
        result = run_comparison_ui("We use cookies.", None, "We use cookies.", None,
                                   "tfidf", None, 0.70)
        self.assertIn("provisional", result[1])
        self.assertIn("0.400", result[1])

    def test_calibration_invalid_configuration(self):
        with self.assertRaises(PolicyDeltaError):
            use_alignment_calibration({"method": "minilm", "threshold": float("nan")})
        with self.assertRaises(PolicyDeltaError):
            alignment_threshold("invalid")




class HistoricalDatasetTests(unittest.TestCase):
    def _records(self):
        return [
            {"site": "alpha", "pair_id": "alpha-2025-2026",
             "old": "Data retention\n\nWe retain account records for 30 days. We protect the data. ",
             "new": "Data retention\n\nWe retain account records for 90 days. We protect the data. ",
             "old_url": "https://example.org/alpha/old", "new_url": "https://example.org/alpha/new"},
            {"site": "beta", "pair_id": "beta-2025-2026",
             "old": "Your rights\n\nYou can request deletion of information. We delete accounts.",
             "new": "Your rights\n\nYou can request removal of information. We delete accounts.",
             "old_url": "https://example.org/beta/old", "new_url": "https://example.org/beta/new"},
        ]

    def test_version_candidates_sort_and_skip_duplicates(self):
        def entry(name, sha):
            return {"name": name, "sha": sha, "size": 2000, "type": "file"}
        versions = [entry("2025-02-01T01-00-00Z.md", "a"),
                    entry("2025-01-01T01-00-00Z.md", "a"),
                    entry("2025-03-01T01-00-00Z.md", "b")]
        result = _version_candidates(versions)
        self.assertEqual(len(result), 1)
        self.assertEqual(result[0][1]["sha"], "b")

    def test_download_verifies_git_hash(self):
        data = b"This is a real UTF-8 stream format used in the input contract."
        git_hash = hashlib.sha1(f"blob {len(data)}\0".encode() + data).hexdigest()
        class Response:
            content = data
            def raise_for_status(self): pass
        class Session:
            def get(self, *args, **kwargs): return Response()
        entry = {"download_url": "https://raw.githubusercontent.com/OpenTermsArchive/contrib-versions/main/example.md",
                 "size": len(data), "sha": git_hash}
        self.assertEqual(_download_version(entry, Session()), data.decode())
        with self.assertRaises(PolicyDeltaError):
            _download_version({**entry, "sha": "wrong"}, Session())

    def test_provenance_and_evidence_are_exported(self):
        with tempfile.TemporaryDirectory() as folder:
            result = _save_historical_pairs(self._records(), Path(folder), revision="b" * 40)
            self.assertEqual(result["pair_count"], 2)
            self.assertEqual(result["calibration_sites"], 1)
            self.assertEqual(result["test_sites"], 1)
            with (Path(folder)/"extraction_results.csv").open() as f:
                rows = list(csv.DictReader(f))
            self.assertTrue(all(r["source_offsets_valid"] == "True" for r in rows))
            self.assertFalse((Path(folder)/"manifest.csv").exists())
            self.assertEqual(len(json.loads((Path(folder)/"provenance.json").read_text())), 2)

    def test_incomplete_review_refuses_gold_manifest(self):
        with tempfile.TemporaryDirectory() as folder:
            _save_historical_pairs(self._records(), Path(folder), revision="b" * 40)
            with self.assertRaises(PolicyDeltaError):
                finalize_reviewed_manifest(folder)
            self.assertFalse((Path(folder)/"manifest.csv").exists())

    def test_complete_review_can_load_and_evaluate(self):
        with tempfile.TemporaryDirectory() as folder:
            _save_historical_pairs(self._records(), Path(folder), revision="b" * 40)
            path = Path(folder)/"review_decisions.csv"
            with path.open(encoding="utf-8", newline="") as f:
                rows = list(csv.DictReader(f))
            for row in rows:
                row["matched_new_index"] = row["old_index"]
                row["reviewed"] = "1"
            with path.open("w", encoding="utf-8", newline="") as f:
                writer=csv.DictWriter(f, fieldnames=rows[0].keys());writer.writeheader();writer.writerows(rows)
            manifest = finalize_reviewed_manifest(folder)
            pairs=load_alignment_benchmark(manifest)
            calibration = calibrate_alignment_threshold(pairs, method="tfidf")
            heldout = evaluate_alignment_threshold(pairs, calibration)
            self.assertEqual(len(pairs), 2)
            self.assertEqual(heldout["test_pairs"], 1)

    def test_full_github_retrieval_with_mocked_responses(self):
        revision = "c" * 40
        old_text = ("Data collection\n\n" + "We collect contact details to provide our service. " * 12)
        new_text = ("Data collection\n\n" + "We collect account details to provide our service. " * 12)
        def version(provider, day, content):
            raw = content.encode("utf-8")
            blob = hashlib.sha1(f"blob {len(raw)}\0".encode() + raw).hexdigest()
            return {"name": f"2025-01-{day}T00-00-00Z.md", "type": "file", "size": len(raw),
                    "sha": blob, "download_url": f"https://raw.githubusercontent.com/{OTA_REPOSITORY}/{revision}/{provider}/{day}.md"}
        versions = {provider: [version(provider, "01", old_text), version(provider, "02", new_text)]
                    for provider in ("Airbnb", "Apple")}
        class Response:
            def __init__(self, *, payload=None, content=b""):
                self._payload, self.content, self.status_code = payload, content, 200
            def json(self): return self._payload
            def raise_for_status(self): pass
        class Session:
            def get(self, url, **kwargs):
                if url.endswith("/commits/main"):
                    return Response(payload={"sha": revision})
                if url.endswith(f"/contents?ref={revision}"):
                    return Response(payload=[{"name": name, "type": "dir"} for name in versions])
                for provider in versions:
                    if url.endswith(f"/contents/{provider}?ref={revision}"):
                        return Response(payload=[{"name": "Privacy Policy", "type": "dir",
                                                  "path": f"{provider}/Privacy Policy"}])
                    if url.endswith(f"/contents/{provider}/Privacy%20Policy?ref={revision}"):
                        return Response(payload=versions[provider])
                    for entry, content in zip(versions[provider], (old_text, new_text)):
                        if url == entry["download_url"]:
                            return Response(content=content.encode("utf-8"))
                raise AssertionError(f"Unexpected request {url}")
        with tempfile.TemporaryDirectory() as folder:
            result = prepare_historical_policies(folder, providers=("Airbnb", "Apple"),
                                                 max_pairs=2, session=Session())
            self.assertEqual(result["pair_count"], 2)
            self.assertTrue((Path(folder)/"review_suggestions.csv").exists())
            self.assertEqual(result["source_revision"], revision)

    def test_review_rejects_modified_source(self):
        with tempfile.TemporaryDirectory() as folder:
            _save_historical_pairs(self._records(), Path(folder), revision="b" * 40)
            source = Path(folder) / "policies" / "alpha-2025-2026-old.md"
            source.write_text(source.read_text() + " An unauthorized change.", encoding="utf-8")
            with self.assertRaises(EvidenceError):
                finalize_reviewed_manifest(folder)
            self.assertFalse((Path(folder) / "manifest.csv").exists())

    def test_review_rejects_duplicate_new_clause(self):
        with tempfile.TemporaryDirectory() as folder:
            _save_historical_pairs(self._records(), Path(folder), revision="b" * 40)
            path = Path(folder)/"review_decisions.csv"
            with path.open(encoding="utf-8", newline="") as f:
                rows=list(csv.DictReader(f))
            for row in rows:
                row["reviewed"] = "1"
                row["matched_new_index"] = "0"
            with path.open("w", encoding="utf-8", newline="") as f:
                writer=csv.DictWriter(f,fieldnames=rows[0].keys());writer.writeheader();writer.writerows(rows)
            with self.assertRaises(PolicyDeltaError):
                finalize_reviewed_manifest(folder)


suite = unittest.TestSuite([
    unittest.defaultTestLoader.loadTestsFromTestCase(PolicyDeltaTests),
    unittest.defaultTestLoader.loadTestsFromTestCase(HistoricalDatasetTests),
])
result = unittest.TextTestRunner(verbosity=1).run(suite)
if not result.wasSuccessful():
    raise RuntimeError("PolicyDelta validation failed.")
print(f"Validation passed: {result.testsRun} tests")

## Application

Running this cell starts the interface. Google Colab may create a public share link; use non-confidential documents.

In [ ]:
demo = build_policy_delta_app()
demo.launch(show_error=True)